<a href="https://colab.research.google.com/github/martintreforedwards-star/playtheroute/blob/main/c2c_GA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# THE ROUTE - C2C MASTER ENRICHMENT
# ==========================================

import pandas as pd
import numpy as np

# Load workbook
df = pd.read_excel("c2c_master.xlsx")

# --------------------------------------------------
# Add missing columns if needed
# --------------------------------------------------

new_cols = [
    "station_id",
    "route",
    "terminus",
    "interchange",
    "latitude",
    "longitude",
    "country",
    "distance_band",
    "coastal",
    "zone",
    "county"
]

for col in new_cols:
    if col not in df.columns:
        df[col] = None

# --------------------------------------------------
# station_id
# --------------------------------------------------

df["station_id"] = [
    f"c2c_{str(i+1).zfill(3)}"
    for i in range(len(df))
]

# --------------------------------------------------
# Route mappings
# --------------------------------------------------

southend_line = {
    "London Fenchurch Street",
    "Limehouse",
    "West Ham",
    "Barking",
    "Upminster",
    "Dagenham Dock",
    "Rainham",
    "Purfleet",
    "Pitsea",
    "Laindon",
    "Basildon",
    "Benfleet",
    "Leigh-on-Sea",
    "Chalkwell",
    "Westcliff",
    "Southend Central",
    "Thorpe Bay",
    "Shoeburyness"
}

tilbury_loop = {
    "Grays",
    "Tilbury Town",
    "East Tilbury",
    "Stanford-le-Hope"
}

ockendon_branch = {
    "Chafford Hundred Lakeside",
    "Ockendon"
}

romford_branch = {
    "Emerson Park",
    "Romford"
}

def get_route(station):

    if station in southend_line:
        return "Southend Line"

    if station in tilbury_loop:
        return "Tilbury Loop"

    if station in ockendon_branch:
        return "Ockendon Branch"

    if station in romford_branch:
        return "Romford Branch"

    return "Unknown"

df["route"] = df["station_name"].apply(get_route)

# --------------------------------------------------
# Terminus
# --------------------------------------------------

terminus_stations = {
    "London Fenchurch Street",
    "Shoeburyness",
    "Ockendon",
    "Romford"
}

df["terminus"] = df["station_name"].isin(terminus_stations)

# --------------------------------------------------
# Interchange
# --------------------------------------------------

interchanges = {
    "London Fenchurch Street",
    "West Ham",
    "Barking",
    "Upminster",
    "Romford",
    "Basildon",
    "Southend Central"
}

df["interchange"] = df["station_name"].isin(interchanges)

# --------------------------------------------------
# Country
# --------------------------------------------------

df["country"] = "England"

# --------------------------------------------------
# Coastal
# --------------------------------------------------

coastal_stations = {
    "Benfleet",
    "Leigh-on-Sea",
    "Chalkwell",
    "Westcliff",
    "Southend Central",
    "Thorpe Bay",
    "Shoeburyness"
}

df["coastal"] = df["station_name"].isin(coastal_stations)

# --------------------------------------------------
# Zone
# --------------------------------------------------

london = {
    "London Fenchurch Street",
    "Limehouse",
    "West Ham",
    "Barking",
    "Upminster",
    "Dagenham Dock",
    "Rainham",
    "Emerson Park",
    "Romford"
}

thurrock = {
    "Purfleet",
    "Chafford Hundred Lakeside",
    "Ockendon",
    "Grays",
    "Tilbury Town",
    "East Tilbury",
    "Stanford-le-Hope"
}

basildon_zone = {
    "Pitsea",
    "Laindon",
    "Basildon"
}

southend_zone = {
    "Benfleet",
    "Leigh-on-Sea",
    "Chalkwell",
    "Westcliff",
    "Southend Central",
    "Thorpe Bay",
    "Shoeburyness"
}

def get_zone(station):

    if station in london:
        return "London"

    if station in thurrock:
        return "Thurrock"

    if station in basildon_zone:
        return "Basildon"

    if station in southend_zone:
        return "Southend"

    return "Essex"

df["zone"] = df["station_name"].apply(get_zone)

# --------------------------------------------------
# County
# --------------------------------------------------

def get_county(station):

    if station in london:
        return "Greater London"

    if station in thurrock:
        return "Thurrock"

    if station in southend_zone:
        return "Southend-on-Sea"

    return "Essex"

df["county"] = df["station_name"].apply(get_county)

# --------------------------------------------------
# Save
# --------------------------------------------------

df.to_excel(
    "c2c_master_v1.xlsx",
    index=False
)

print("Saved: c2c_master_v1.xlsx")
print(df.head())

Saved: c2c_master_v1.xlsx
              station_name crs_code operator  fenchurch_minutes  \
0  London Fenchurch Street      FST      c2c                NaN   
1                Limehouse      LHS      c2c                NaN   
2                 West Ham      WEH      c2c                NaN   
3                  Barking      BKG      c2c                NaN   
4                Upminster      UPM      c2c                NaN   

   distance_from_fenchurch_km station_id          route  terminus  \
0                         NaN    c2c_001  Southend Line      True   
1                         NaN    c2c_002  Southend Line     False   
2                         NaN    c2c_003  Southend Line     False   
3                         NaN    c2c_004  Southend Line     False   
4                         NaN    c2c_005  Southend Line     False   

   interchange latitude longitude  country distance_band  coastal    zone  \
0         True     None      None  England          None    False  London   
1 

In [ ]:
# ==========================================
# THE ROUTE - C2C COORDINATES + DISTANCES
# ==========================================

import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, atan2

# ----------------------------
# Load files
# ----------------------------

c2c = pd.read_excel("c2c_master_v1.xlsx")

stations = pd.read_csv("station.csv")

# ----------------------------
# Match CRS codes
# ----------------------------

stations_lookup = stations.set_index("crsCode")

c2c["latitude"] = c2c["crs_code"].map(
    stations_lookup["lat"]
)

c2c["longitude"] = c2c["crs_code"].map(
    stations_lookup["long"]
)

# ----------------------------
# Optional country refresh
# ----------------------------

country_map = {
    "england": "England",
    "wales": "Wales",
    "scotland": "Scotland"
}

c2c["country"] = (
    c2c["crs_code"]
    .map(stations_lookup["constituentCountry"])
    .map(country_map)
    .fillna("England")
)

# ----------------------------
# Fenchurch Street coordinates
# ----------------------------

fenchurch = c2c[
    c2c["station_name"] == "London Fenchurch Street"
].iloc[0]

origin_lat = fenchurch["latitude"]
origin_lon = fenchurch["longitude"]

# ----------------------------
# Haversine distance function
# ----------------------------

def haversine(lat1, lon1, lat2, lon2):

    R = 6371.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        sin(dlat / 2) ** 2
        + cos(lat1)
        * cos(lat2)
        * sin(dlon / 2) ** 2
    )

    c = 2 * atan2(
        sqrt(a),
        sqrt(1 - a)
    )

    return R * c

# ----------------------------
# Distance from Fenchurch Street
# ----------------------------

c2c["distance_from_fenchurch_km"] = c2c.apply(
    lambda row: round(
        haversine(
            origin_lat,
            origin_lon,
            row["latitude"],
            row["longitude"]
        ),
        1
    ),
    axis=1
)

# ----------------------------
# Distance bands
# ----------------------------

def distance_band(km):

    if km <= 20:
        return "local"

    elif km <= 50:
        return "regional"

    else:
        return "long"

c2c["distance_band"] = (
    c2c["distance_from_fenchurch_km"]
    .apply(distance_band)
)

# ----------------------------
# Save output
# ----------------------------

c2c.to_excel(
    "c2c_master_v2.xlsx",
    index=False
)

# ----------------------------
# Validation output
# ----------------------------

print("Saved: c2c_master_v2.xlsx")
print()

print(
    c2c[
        [
            "station_name",
            "crs_code",
            "latitude",
            "longitude",
            "distance_from_fenchurch_km",
            "distance_band"
        ]
    ].head()
)

print()
print("Stations:", len(c2c))
print(
    "Missing coordinates:",
    c2c["latitude"].isna().sum()
)

FileNotFoundError: [Errno 2] No such file or directory: 'station.csv'

In [ ]:
 ---------------------------------------------------------------------------
FileNotFoundError                         Traceback (most recent call last)
/tmp/ipykernel_3299/3029985776.py in <cell line: 0>()
     13 c2c = pd.read_excel("c2c_master_v1.xlsx")
     14
---> 15 stations = pd.read_csv("station.csv")
     16
     17 # ----------------------------

4 frames/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py in read_csv(filepath_or_buffer, sep, delimiter, header, names, index_col, usecols, dtype, engine, converters, true_values, false_values, skipinitialspace, skiprows, skipfooter, nrows, na_values, keep_default_na, na_filter, verbose, skip_blank_lines, parse_dates, infer_datetime_format, keep_date_col, date_parser, date_format, dayfirst, cache_dates, iterator, chunksize, compression, thousands, decimal, lineterminator, quotechar, quoting, doublequote, escapechar, comment, encoding, encoding_errors, dialect, on_bad_lines, delim_whitespace, low_memory, memory_map, float_precision, storage_options, dtype_backend)
   1024     kwds.update(kwds_defaults)
   1025
-> 1026     return _read(filepath_or_buffer, kwds)
   1027
   1028

/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py in _read(filepath_or_buffer, kwds)
    618
    619     # Create the parser.
--> 620     parser = TextFileReader(filepath_or_buffer, **kwds)
    621
    622     if chunksize or iterator:

/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py in __init__(self, f, engine, **kwds)
   1618
   1619         self.handles: IOHandles | None = None
-> 1620         self._engine = self._make_engine(f, self.engine)
   1621
   1622     def close(self) -> None:

/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py in _make_engine(self, f, engine)
   1878                 if "b" not in mode:
   1879                     mode += "b"
-> 1880             self.handles = get_handle(
   1881                 f,
   1882                 mode,

/usr/local/lib/python3.12/dist-packages/pandas/io/common.py in get_handle(path_or_buf, mode, encoding, compression, memory_map, is_text, errors, storage_options)
    871         if ioargs.encoding and "b" not in ioargs.mode:
    872             # Encoding
--> 873             handle = open(
    874                 handle,
    875                 ioargs.mode,

FileNotFoundError: [Errno 2] No such file or directory: 'station.csv'

In [ ]:
 ---------------------------------------------------------------------------
FileNotFoundError                         Traceback (most recent call last)
/tmp/ipykernel_3299/3029985776.py in <cell line: 0>()
     13 c2c = pd.read_excel("c2c_master_v1.xlsx")
     14
---> 15 stations = pd.read_csv("station.csv")
     16
     17 # ----------------------------

4 frames/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py in read_csv(filepath_or_buffer, sep, delimiter, header, names, index_col, usecols, dtype, engine, converters, true_values, false_values, skipinitialspace, skiprows, skipfooter, nrows, na_values, keep_default_na, na_filter, verbose, skip_blank_lines, parse_dates, infer_datetime_format, keep_date_col, date_parser, date_format, dayfirst, cache_dates, iterator, chunksize, compression, thousands, decimal, lineterminator, quotechar, quoting, doublequote, escapechar, comment, encoding, encoding_errors, dialect, on_bad_lines, delim_whitespace, low_memory, memory_map, float_precision, storage_options, dtype_backend)
   1024     kwds.update(kwds_defaults)
   1025
-> 1026     return _read(filepath_or_buffer, kwds)
   1027
   1028

/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py in _read(filepath_or_buffer, kwds)
    618
    619     # Create the parser.
--> 620     parser = TextFileReader(filepath_or_buffer, **kwds)
    621
    622     if chunksize or iterator:

/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py in __init__(self, f, engine, **kwds)
   1618
   1619         self.handles: IOHandles | None = None
-> 1620         self._engine = self._make_engine(f, self.engine)
   1621
   1622     def close(self) -> None:

/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py in _make_engine(self, f, engine)
   1878                 if "b" not in mode:
   1879                     mode += "b"
-> 1880             self.handles = get_handle(
   1881                 f,
   1882                 mode,

/usr/local/lib/python3.12/dist-packages/pandas/io/common.py in get_handle(path_or_buf, mode, encoding, compression, memory_map, is_text, errors, storage_options)
    871         if ioargs.encoding and "b" not in ioargs.mode:
    872             # Encoding
--> 873             handle = open(
    874                 handle,
    875                 ioargs.mode,

FileNotFoundError: [Errno 2] No such file or directory: 'station.csv'

In [ ]:
# ==========================================
# THE ROUTE - C2C COORDINATES + DISTANCES
# ==========================================

import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, atan2

# ----------------------------
# Load files
# ----------------------------

c2c = pd.read_excel("c2c_master_v1.xlsx")

stations = pd.read_csv("stations.csv")

# ----------------------------
# Match CRS codes
# ----------------------------

stations_lookup = stations.set_index("crsCode")

c2c["latitude"] = c2c["crs_code"].map(
    stations_lookup["lat"]
)

c2c["longitude"] = c2c["crs_code"].map(
    stations_lookup["long"]
)

# ----------------------------
# Optional country refresh
# ----------------------------

country_map = {
    "england": "England",
    "wales": "Wales",
    "scotland": "Scotland"
}

c2c["country"] = (
    c2c["crs_code"]
    .map(stations_lookup["constituentCountry"])
    .map(country_map)
    .fillna("England")
)

# ----------------------------
# Fenchurch Street coordinates
# ----------------------------

fenchurch = c2c[
    c2c["station_name"] == "London Fenchurch Street"
].iloc[0]

origin_lat = fenchurch["latitude"]
origin_lon = fenchurch["longitude"]

# ----------------------------
# Haversine distance function
# ----------------------------

def haversine(lat1, lon1, lat2, lon2):

    R = 6371.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        sin(dlat / 2) ** 2
        + cos(lat1)
        * cos(lat2)
        * sin(dlon / 2) ** 2
    )

    c = 2 * atan2(
        sqrt(a),
        sqrt(1 - a)
    )

    return R * c

# ----------------------------
# Distance from Fenchurch Street
# ----------------------------

c2c["distance_from_fenchurch_km"] = c2c.apply(
    lambda row: round(
        haversine(
            origin_lat,
            origin_lon,
            row["latitude"],
            row["longitude"]
        ),
        1
    ),
    axis=1
)

# ----------------------------
# Distance bands
# ----------------------------

def distance_band(km):

    if km <= 20:
        return "local"

    elif km <= 50:
        return "regional"

    else:
        return "long"

c2c["distance_band"] = (
    c2c["distance_from_fenchurch_km"]
    .apply(distance_band)
)

# ----------------------------
# Save output
# ----------------------------

c2c.to_excel(
    "c2c_master_v2.xlsx",
    index=False
)

# ----------------------------
# Validation output
# ----------------------------

print("Saved: c2c_master_v2.xlsx")
print()

print(
    c2c[
        [
            "station_name",
            "crs_code",
            "latitude",
            "longitude",
            "distance_from_fenchurch_km",
            "distance_band"
        ]
    ].head()
)

print()
print("Stations:", len(c2c))
print(
    "Missing coordinates:",
    c2c["latitude"].isna().sum()
)

Saved: c2c_master_v2.xlsx

              station_name crs_code   latitude  longitude  \
0  London Fenchurch Street      FST  51.511234  -0.079039   
1                Limehouse      LHS  51.512390  -0.040071   
2                 West Ham      WEH  51.528709   0.005323   
3                  Barking      BKG  51.539982   0.080804   
4                Upminster      UPM  51.559307   0.251938   

   distance_from_fenchurch_km distance_band  
0                         0.0         local  
1                         2.7         local  
2                         6.2         local  
3                        11.5         local  
4                        23.5      regional  

Stations: 26
Missing coordinates: 0


In [ ]:
import pandas as pd

df = pd.read_excel("c2c_master_v2.xlsx")

minutes_lookup = {

    "London Fenchurch Street": 0,
    "Limehouse": 4,
    "West Ham": 8,
    "Barking": 12,
    "Dagenham Dock": 16,
    "Rainham": 19,
    "Upminster": 22,

    "Purfleet": 25,
    "Chafford Hundred Lakeside": 28,
    "Ockendon": 32,

    "Grays": 30,
    "Tilbury Town": 35,
    "East Tilbury": 40,
    "Stanford-le-Hope": 45,

    "Laindon": 28,
    "Basildon": 32,
    "Pitsea": 35,

    "Benfleet": 40,
    "Leigh-on-Sea": 44,
    "Chalkwell": 46,
    "Westcliff": 48,
    "Southend Central": 50,
    "Thorpe Bay": 54,
    "Shoeburyness": 58,

    "Emerson Park": 25,
    "Romford": 29
}

df["fenchurch_minutes"] = (
    df["station_name"]
    .map(minutes_lookup)
)

print(
    "Missing timings:",
    df["fenchurch_minutes"].isna().sum()
)

df.to_excel(
    "c2c_master_v3.xlsx",
    index=False
)

print("Saved: c2c_master_v3.xlsx")

Missing timings: 0
Saved: c2c_master_v3.xlsx


In [ ]:
import pandas as pd

df = pd.read_excel("c2c_master_v3.xlsx")

print(df.info())

print("\nRoutes")
print(df["route"].value_counts())

print("\nDistance Bands")
print(df["distance_band"].value_counts())

print("\nMissing Values")
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26 entries, 0 to 25
Data columns (total 16 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   station_name                26 non-null     object 
 1   crs_code                    26 non-null     object 
 2   operator                    26 non-null     object 
 3   fenchurch_minutes           26 non-null     int64  
 4   distance_from_fenchurch_km  26 non-null     float64
 5   station_id                  26 non-null     object 
 6   route                       26 non-null     object 
 7   terminus                    26 non-null     bool   
 8   interchange                 26 non-null     bool   
 9   latitude                    26 non-null     float64
 10  longitude                   26 non-null     float64
 11  country                     26 non-null     object 
 12  distance_band               26 non-null     object 
 13  coastal                     26 non-nu

In [ ]:
import pandas as pd

df = pd.read_excel("greateranglia_master.xlsx")

print(df.columns.tolist())
print()
print("Stations:", len(df))
print()
print(df.head())


['station_name', 'crs_code', 'operator']

Stations: 26

              station_name crs_code        operator
0  London Liverpool Street      LST  Greater Anglia
1                Stratford      SRA  Greater Anglia
2                 Maryland      MYL  Greater Anglia
3              Forest Gate      FOG  Greater Anglia
4               Manor Park      MNP  Greater Anglia


In [ ]:
# ==========================================
# THE ROUTE - GREATER ANGLIA METADATA
# ==========================================

import pandas as pd

# ----------------------------
# Load workbook
# ----------------------------

df = pd.read_excel("greateranglia_master.xlsx")

# ----------------------------
# Add missing columns
# ----------------------------

new_cols = [
    "station_id",
    "route",
    "terminus",
    "interchange",
    "latitude",
    "longitude",
    "country",
    "distance_band",
    "coastal",
    "zone",
    "county",
    "liverpool_street_minutes",
    "distance_from_liverpool_street_km"
]

for col in new_cols:
    if col not in df.columns:
        df[col] = None

# ----------------------------
# station_id
# ----------------------------

df["station_id"] = [
    f"ga_{str(i+1).zfill(3)}"
    for i in range(len(df))
]

# ----------------------------
# Route
# ----------------------------

df["route"] = "Great Eastern Main Line"

# ----------------------------
# Terminus
# ----------------------------

terminus_stations = {
    "London Liverpool Street",
    "Norwich"
}

df["terminus"] = df["station_name"].isin(
    terminus_stations
)

# ----------------------------
# Interchange
# ----------------------------

interchanges = {
    "London Liverpool Street",
    "Stratford",
    "Romford",
    "Shenfield",
    "Chelmsford",
    "Colchester",
    "Ipswich",
    "Norwich"
}

df["interchange"] = df["station_name"].isin(
    interchanges
)

# ----------------------------
# Country
# ----------------------------

df["country"] = "England"

# ----------------------------
# Coastal
# ----------------------------

df["coastal"] = False

# ----------------------------
# Zones
# ----------------------------

london = {
    "London Liverpool Street",
    "Stratford",
    "Maryland",
    "Forest Gate",
    "Manor Park",
    "Ilford",
    "Seven Kings",
    "Goodmayes",
    "Chadwell Heath",
    "Romford",
    "Gidea Park",
    "Harold Wood"
}

essex = {
    "Brentwood",
    "Shenfield",
    "Ingatestone",
    "Chelmsford",
    "Hatfield Peverel",
    "Witham",
    "Kelvedon",
    "Marks Tey",
    "Colchester",
    "Manningtree"
}

suffolk = {
    "Ipswich",
    "Stowmarket"
}

norfolk = {
    "Diss",
    "Norwich"
}

def get_zone(station):

    if station in london:
        return "London"

    if station in essex:
        return "Essex"

    if station in suffolk:
        return "Suffolk"

    if station in norfolk:
        return "Norfolk"

    return "Unknown"

df["zone"] = df["station_name"].apply(get_zone)

# ----------------------------
# County
# ----------------------------

def get_county(station):

    if station in london:
        return "Greater London"

    if station in essex:
        return "Essex"

    if station in suffolk:
        return "Suffolk"

    if station in norfolk:
        return "Norfolk"

    return "Unknown"

df["county"] = df["station_name"].apply(get_county)

# ----------------------------
# Save
# ----------------------------

df.to_excel(
    "greateranglia_master_v1.xlsx",
    index=False
)

print("Saved: greateranglia_master_v1.xlsx")
print()
print(df.head())

Saved: greateranglia_master_v1.xlsx

              station_name crs_code        operator station_id  \
0  London Liverpool Street      LST  Greater Anglia     ga_001   
1                Stratford      SRA  Greater Anglia     ga_002   
2                 Maryland      MYL  Greater Anglia     ga_003   
3              Forest Gate      FOG  Greater Anglia     ga_004   
4               Manor Park      MNP  Greater Anglia     ga_005   

                     route  terminus  interchange latitude longitude  country  \
0  Great Eastern Main Line      True         True     None      None  England   
1  Great Eastern Main Line     False         True     None      None  England   
2  Great Eastern Main Line     False        False     None      None  England   
3  Great Eastern Main Line     False        False     None      None  England   
4  Great Eastern Main Line     False        False     None      None  England   

  distance_band  coastal    zone          county liverpool_street_minutes  \
0 

In [ ]:
# ==========================================
# THE ROUTE - GREATER ANGLIA COORDINATES
# ==========================================

import pandas as pd
from math import radians, sin, cos, sqrt, atan2

# ----------------------------
# Load files
# ----------------------------

ga = pd.read_excel("greateranglia_master_v1.xlsx")

stations = pd.read_csv("stations.csv")

# ----------------------------
# Build lookup
# ----------------------------

stations_lookup = stations.set_index("crsCode")

ga["latitude"] = ga["crs_code"].map(
    stations_lookup["lat"]
)

ga["longitude"] = ga["crs_code"].map(
    stations_lookup["long"]
)

# ----------------------------
# Liverpool Street origin
# ----------------------------

origin = ga[
    ga["station_name"] == "London Liverpool Street"
].iloc[0]

origin_lat = origin["latitude"]
origin_lon = origin["longitude"]

# ----------------------------
# Haversine formula
# ----------------------------

def haversine(lat1, lon1, lat2, lon2):

    R = 6371.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        sin(dlat / 2) ** 2
        + cos(lat1)
        * cos(lat2)
        * sin(dlon / 2) ** 2
    )

    c = 2 * atan2(
        sqrt(a),
        sqrt(1 - a)
    )

    return R * c

# ----------------------------
# Distances
# ----------------------------

ga["distance_from_liverpool_street_km"] = ga.apply(
    lambda row: round(
        haversine(
            origin_lat,
            origin_lon,
            row["latitude"],
            row["longitude"]
        ),
        1
    ),
    axis=1
)

# ----------------------------
# Distance bands
# ----------------------------

def distance_band(km):

    if km <= 20:
        return "local"

    elif km <= 80:
        return "regional"

    else:
        return "long"

ga["distance_band"] = (
    ga["distance_from_liverpool_street_km"]
    .apply(distance_band)
)

# ----------------------------
# Save
# ----------------------------

ga.to_excel(
    "greateranglia_master_v2.xlsx",
    index=False
)

print("Saved: greateranglia_master_v2.xlsx")
print()

print(
    ga[
        [
            "station_name",
            "crs_code",
            "latitude",
            "longitude",
            "distance_from_liverpool_street_km",
            "distance_band"
        ]
    ].head()
)

print()
print("Stations:", len(ga))
print(
    "Missing coordinates:",
    ga["latitude"].isna().sum()
)

Saved: greateranglia_master_v2.xlsx

              station_name crs_code   latitude  longitude  \
0  London Liverpool Street      LST  51.517551  -0.080210   
1                Stratford      SRA  51.542358  -0.004175   
2                 Maryland      MYL  51.545784   0.006075   
3              Forest Gate      FOG  51.549976   0.023568   
4               Manor Park      MNP  51.552296   0.045309   

   distance_from_liverpool_street_km distance_band  
0                                0.0         local  
1                                5.9         local  
2                                6.7         local  
3                                8.0         local  
4                                9.5         local  

Stations: 26
Missing coordinates: 1


In [ ]:
import pandas as pd

ga = pd.read_excel("greateranglia_master_v2.xlsx")

missing = ga[ga["latitude"].isna()]

print("Missing stations:", len(missing))
print()
print(missing.to_string(index=False))

Missing stations: 1

station_name crs_code       operator station_id                   route  terminus  interchange  latitude  longitude country distance_band  coastal  zone county  liverpool_street_minutes  distance_from_liverpool_street_km
   Marks Tey      MRK Greater Anglia     ga_020 Great Eastern Main Line     False        False       NaN        NaN England          long    False Essex  Essex                       NaN                                NaN


In [ ]:
import pandas as pd

stations = pd.read_csv("stations.csv")

matches = stations[
    stations["stationName"]
    .str.contains("Marks", case=False, na=False)
]

print(matches[
    ["stationName", "crsCode", "lat", "long"]
].to_string(index=False))

stationName crsCode       lat     long
  Marks Tey     MKT 51.880695 0.782372


In [ ]:
import pandas as pd

ga = pd.read_excel("greateranglia_master_v2.xlsx")

ga.loc[
    ga["station_name"] == "Marks Tey",
    "crs_code"
] = "MKT"

ga.to_excel(
    "greateranglia_master_v2.xlsx",
    index=False
)

print("Marks Tey CRS corrected to MKT")

Marks Tey CRS corrected to MKT


In [ ]:
ga = pd.read_excel("greateranglia_master_v2.xlsx")

ga.loc[
    ga["station_name"] == "Marks Tey",
    "latitude"
] = <lat>

ga.loc[
    ga["station_name"] == "Marks Tey",
    "longitude"
] = <lon>

SyntaxError: invalid syntax (166699853.py, line 6)

In [ ]:
import pandas as pd

ga = pd.read_excel("greateranglia_master_v2.xlsx")

ga.loc[
    ga["station_name"] == "Marks Tey",
    "crs_code"
] = "MKT"

ga.to_excel(
    "greateranglia_master_v2.xlsx",
    index=False
)

print("Marks Tey CRS corrected to MKT")

Marks Tey CRS corrected to MKT


In [ ]:
# ==========================================
# THE ROUTE - GREATER ANGLIA COORDINATES
# ==========================================

import pandas as pd
from math import radians, sin, cos, sqrt, atan2

# ----------------------------
# Load files
# ----------------------------

ga = pd.read_excel("greateranglia_master_v2.xlsx")

stations = pd.read_csv("stations.csv")

# ----------------------------
# CRS lookup
# ----------------------------

stations_lookup = stations.set_index("crsCode")

ga["latitude"] = ga["crs_code"].map(
    stations_lookup["lat"]
)

ga["longitude"] = ga["crs_code"].map(
    stations_lookup["long"]
)

# ----------------------------
# Liverpool Street origin
# ----------------------------

origin = ga[
    ga["station_name"] == "London Liverpool Street"
].iloc[0]

origin_lat = origin["latitude"]
origin_lon = origin["longitude"]

# ----------------------------
# Haversine distance function
# ----------------------------

def haversine(lat1, lon1, lat2, lon2):

    R = 6371.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        sin(dlat / 2) ** 2
        + cos(lat1)
        * cos(lat2)
        * sin(dlon / 2) ** 2
    )

    c = 2 * atan2(
        sqrt(a),
        sqrt(1 - a)
    )

    return R * c

# ----------------------------
# Calculate distances
# ----------------------------

ga["distance_from_liverpool_street_km"] = ga.apply(
    lambda row: round(
        haversine(
            origin_lat,
            origin_lon,
            row["latitude"],
            row["longitude"]
        ),
        1
    ),
    axis=1
)

# ----------------------------
# Distance bands
# ----------------------------

def distance_band(km):

    if km <= 20:
        return "local"
    elif km <= 80:
        return "regional"
    else:
        return "long"

ga["distance_band"] = ga[
    "distance_from_liverpool_street_km"
].apply(distance_band)

# ----------------------------
# Save
# ----------------------------

ga.to_excel(
    "greateranglia_master_v2.xlsx",
    index=False
)

# ----------------------------
# Validation
# ----------------------------

print("Saved: greateranglia_master_v2.xlsx")
print()

print(
    ga[
        [
            "station_name",
            "crs_code",
            "latitude",
            "longitude",
            "distance_from_liverpool_street_km",
            "distance_band"
        ]
    ].tail()
)

print()
print("Stations:", len(ga))
print("Missing coordinates:", ga["latitude"].isna().sum())

Saved: greateranglia_master_v2.xlsx

   station_name crs_code   latitude  longitude  \
21  Manningtree      MNG  51.948818   1.045653   
22      Ipswich      IPS  52.050552   1.144481   
23   Stowmarket      SMK  52.190105   1.000662   
24         Diss      DIS  52.373096   1.123274   
25      Norwich      NRW  52.627121   1.306879   

    distance_from_liverpool_street_km distance_band  
21                               91.2          long  
22                              103.0          long  
23                              105.4          long  
24                              125.9          long  
25                              155.6          long  

Stations: 26
Missing coordinates: 0


In [ ]:
# ==========================================
# THE ROUTE - GREATER ANGLIA TIMINGS
# ==========================================

import pandas as pd

ga = pd.read_excel("greateranglia_master_v2.xlsx")

minutes_lookup = {

    "London Liverpool Street": 0,
    "Stratford": 8,
    "Maryland": 10,
    "Forest Gate": 11,
    "Manor Park": 13,
    "Ilford": 15,
    "Seven Kings": 17,
    "Goodmayes": 18,
    "Chadwell Heath": 20,
    "Romford": 22,
    "Gidea Park": 24,
    "Harold Wood": 26,
    "Brentwood": 29,
    "Shenfield": 32,
    "Ingatestone": 37,
    "Chelmsford": 43,
    "Hatfield Peverel": 48,
    "Witham": 53,
    "Kelvedon": 58,
    "Marks Tey": 62,
    "Colchester": 67,
    "Manningtree": 75,
    "Ipswich": 85,
    "Stowmarket": 95,
    "Diss": 105,
    "Norwich": 115
}

ga["liverpool_street_minutes"] = (
    ga["station_name"]
    .map(minutes_lookup)
)

print(
    "Missing timings:",
    ga["liverpool_street_minutes"].isna().sum()
)

ga.to_excel(
    "greateranglia_master_v3.xlsx",
    index=False
)

print("Saved: greateranglia_master_v3.xlsx")

Missing timings: 0
Saved: greateranglia_master_v3.xlsx


In [ ]:
import pandas as pd

ga = pd.read_excel("greateranglia_master_v3.xlsx")

print(ga.info())

print("\nRoutes")
print(ga["route"].value_counts())

print("\nDistance Bands")
print(ga["distance_band"].value_counts())

print("\nMissing Values")
print(ga.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26 entries, 0 to 25
Data columns (total 16 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   station_name                       26 non-null     object 
 1   crs_code                           26 non-null     object 
 2   operator                           26 non-null     object 
 3   station_id                         26 non-null     object 
 4   route                              26 non-null     object 
 5   terminus                           26 non-null     bool   
 6   interchange                        26 non-null     bool   
 7   latitude                           26 non-null     float64
 8   longitude                          26 non-null     float64
 9   country                            26 non-null     object 
 10  distance_band                      26 non-null     object 
 11  coastal                            26 non-null     bool   
 

In [ ]:
# ==========================================
# THE ROUTE - MASTER MERGE V1
# ==========================================

import pandas as pd

# ----------------------------
# Load datasets
# ----------------------------

gwr = pd.read_csv("gwr_stations_master_v13.csv")

c2c = pd.read_excel("c2c_master_v3.xlsx")

ga = pd.read_excel("greateranglia_master_v3.xlsx")

# ==================================================
# GWR STANDARDISATION
# ==================================================

gwr["operator"] = "GWR"

gwr["origin_station"] = "London Paddington"

gwr["origin_minutes"] = gwr["paddington_minutes"]

gwr["distance_from_origin_km"] = gwr["distance_from_paddington_km"]

# ==================================================
# C2C STANDARDISATION
# ==================================================

c2c["origin_station"] = "London Fenchurch Street"

c2c["origin_minutes"] = c2c["fenchurch_minutes"]

c2c["distance_from_origin_km"] = (
    c2c["distance_from_fenchurch_km"]
)

# Rename CRS column to match GWR
c2c = c2c.rename(
    columns={
        "crs_code": "crs"
    }
)

# ==================================================
# GREATER ANGLIA STANDARDISATION
# ==================================================

ga["origin_station"] = "London Liverpool Street"

ga["origin_minutes"] = ga["liverpool_street_minutes"]

ga["distance_from_origin_km"] = (
    ga["distance_from_liverpool_street_km"]
)

ga = ga.rename(
    columns={
        "crs_code": "crs"
    }
)

# ==================================================
# TARGET COLUMN ORDER
# ==================================================

master_columns = [

    "station_id",
    "station_name",
    "crs",
    "operator",

    "route",

    "origin_station",
    "origin_minutes",
    "distance_from_origin_km",

    "terminus",
    "interchange",

    "latitude",
    "longitude",

    "country",
    "county",
    "zone",

    "distance_band",
    "coastal"
]

# Create any missing columns
for df in [gwr, c2c, ga]:

    for col in master_columns:

        if col not in df.columns:
            df[col] = None

# Reorder
gwr = gwr[master_columns]
c2c = c2c[master_columns]
ga = ga[master_columns]

# ==================================================
# MERGE
# ==================================================

master = pd.concat(
    [
        gwr,
        c2c,
        ga
    ],
    ignore_index=True
)

# ==================================================
# SAVE
# ==================================================

master.to_csv(
    "the_route_master_v1.csv",
    index=False
)

# ==================================================
# QA
# ==================================================

print("MASTER CREATED")
print()

print("Total stations:", len(master))
print()

print("By operator:")
print(master["operator"].value_counts())

print()
print("Missing values:")
print(
    master.isnull().sum()[
        master.isnull().sum() > 0
    ]
)

print()
print(master.head())

FileNotFoundError: [Errno 2] No such file or directory: 'gwr_stations_master_v13.csv'

In [ ]:
import os

for f in sorted(os.listdir()):
    print(f)

.config
c2c_master.xlsx
c2c_master_v1.xlsx
c2c_master_v2.xlsx
c2c_master_v3.xlsx
greateranglia_master.xlsx
greateranglia_master_v1.xlsx
greateranglia_master_v2.xlsx
greateranglia_master_v3.xlsx
sample_data
stations.csv


In [ ]:
# ==========================================
# THE ROUTE - EAST ANGLIA MASTER MERGE
# c2c + Greater Anglia
# ==========================================

import pandas as pd

# ----------------------------
# Load files
# ----------------------------

c2c = pd.read_excel("c2c_master_v3.xlsx")

ga = pd.read_excel("greateranglia_master_v3.xlsx")

# ----------------------------
# Standardise column names
# ----------------------------

c2c = c2c.rename(columns={
    "crs_code": "crs",
    "fenchurch_minutes": "origin_minutes",
    "distance_from_fenchurch_km": "distance_from_origin_km"
})

ga = ga.rename(columns={
    "crs_code": "crs",
    "liverpool_street_minutes": "origin_minutes",
    "distance_from_liverpool_street_km": "distance_from_origin_km"
})

# ----------------------------
# Add origin station
# ----------------------------

c2c["origin_station"] = "London Fenchurch Street"

ga["origin_station"] = "London Liverpool Street"

# ----------------------------
# Common column order
# ----------------------------

master_columns = [
    "station_id",
    "station_name",
    "crs",
    "operator",
    "route",
    "origin_station",
    "origin_minutes",
    "distance_from_origin_km",
    "terminus",
    "interchange",
    "latitude",
    "longitude",
    "country",
    "county",
    "zone",
    "distance_band",
    "coastal"
]

# Ensure both dataframes have all columns
for df in [c2c, ga]:
    for col in master_columns:
        if col not in df.columns:
            df[col] = None

# Reorder columns
c2c = c2c[master_columns]
ga = ga[master_columns]

# ----------------------------
# Merge
# ----------------------------

master = pd.concat(
    [c2c, ga],
    ignore_index=True
)

# ----------------------------
# Save
# ----------------------------

master.to_csv(
    "east_anglia_master_v1.csv",
    index=False
)

master.to_excel(
    "east_anglia_master_v1.xlsx",
    index=False
)

# ----------------------------
# QA
# ----------------------------

print("MASTER CREATED")
print()

print("Total stations:", len(master))
print()

print("By operator:")
print(master["operator"].value_counts())

print()
print("Missing values:")
print(master.isnull().sum())

print()
print("Duplicate station names:",
      master["station_name"].duplicated().sum())

print("Duplicate CRS codes:",
      master["crs"].duplicated().sum())

MASTER CREATED

Total stations: 52

By operator:
operator
c2c               26
Greater Anglia    26
Name: count, dtype: int64

Missing values:
station_id                 0
station_name               0
crs                        0
operator                   0
route                      0
origin_station             0
origin_minutes             0
distance_from_origin_km    0
terminus                   0
interchange                0
latitude                   0
longitude                  0
country                    0
county                     0
zone                       0
distance_band              0
coastal                    0
dtype: int64

Duplicate station names: 1
Duplicate CRS codes: 1


In [ ]:
master = pd.read_csv("east_anglia_master_v1.csv")

dupes = master[
    master["station_name"].duplicated(keep=False)
]

print(
    dupes[
        [
            "station_name",
            "crs",
            "operator",
            "route"
        ]
    ].sort_values("station_name")
)

   station_name  crs        operator                    route
6       Romford  RMF             c2c           Romford Branch
35      Romford  RMF  Greater Anglia  Great Eastern Main Line


In [ ]:
import shutil

shutil.copy(
    "east_anglia_master_v1.xlsx",
    "east_anglia_master_v1_backup.xlsx"
)

print("Backup created")

Backup created


In [ ]:
import pandas as pd

ext = pd.read_excel("greateranglia_extensions.xlsx")

print("Stations:", len(ext))
print(ext.head())

Stations: 36
            station_name crs_code        operator
0             Billericay      BIC  Greater Anglia
1               Wickford      WIC  Greater Anglia
2          Battlesbridge      BSB  Greater Anglia
3  South Woodham Ferrers      SOF  Greater Anglia
4        North Fambridge      NFA  Greater Anglia


In [ ]:
# ==========================================
# THE ROUTE - GREATER ANGLIA EXTENSIONS
# ==========================================

import pandas as pd
from math import radians, sin, cos, sqrt, atan2

# ----------------------------
# Load files
# ----------------------------

ext = pd.read_excel("greateranglia_extensions.xlsx")
stations = pd.read_csv("stations.csv")

# ----------------------------
# Add columns
# ----------------------------

new_cols = [
    "station_id",
    "route",
    "terminus",
    "interchange",
    "latitude",
    "longitude",
    "country",
    "distance_band",
    "coastal",
    "zone",
    "county",
    "liverpool_street_minutes",
    "distance_from_liverpool_street_km"
]

for col in new_cols:
    if col not in ext.columns:
        ext[col] = None

# ----------------------------
# Station IDs
# Continue after ga_026
# ----------------------------

ext["station_id"] = [
    f"ga_{str(i+27).zfill(3)}"
    for i in range(len(ext))
]

# ----------------------------
# Route assignment
# ----------------------------

southend_victoria = {
    "Billericay","Wickford","Rayleigh","Hockley",
    "Rochford","Southend Airport",
    "Prittlewell","Southend Victoria"
}

southminster = {
    "Battlesbridge","South Woodham Ferrers",
    "North Fambridge","Althorne",
    "Burnham-on-Crouch","Southminster"
}

braintree = {
    "White Notley","Cressing",
    "Braintree Freeport","Braintree"
}

clacton_walton = {
    "Hythe","Wivenhoe","Alresford",
    "Great Bentley","Weeley",
    "Thorpe-le-Soken","Clacton-on-Sea",
    "Kirby Cross","Frinton-on-Sea",
    "Walton-on-the-Naze"
}

harwich = {
    "Wrabness","Mistley",
    "Harwich International",
    "Dovercourt","Harwich Town"
}

sudbury = {
    "Chappel and Wakes Colne",
    "Bures","Sudbury"
}

def get_route(station):

    if station in southend_victoria:
        return "Southend Victoria Line"

    if station in southminster:
        return "Southminster Branch"

    if station in braintree:
        return "Braintree Branch"

    if station in clacton_walton:
        return "Clacton/Walton Branch"

    if station in harwich:
        return "Harwich Branch"

    if station in sudbury:
        return "Sudbury Branch"

    return "Unknown"

ext["route"] = ext["station_name"].apply(get_route)

# ----------------------------
# Terminus
# ----------------------------

terminus = {
    "Southend Victoria",
    "Southminster",
    "Braintree",
    "Clacton-on-Sea",
    "Walton-on-the-Naze",
    "Harwich Town",
    "Sudbury"
}

ext["terminus"] = ext["station_name"].isin(terminus)

# ----------------------------
# Interchanges
# ----------------------------

interchanges = {
    "Wickford",
    "Thorpe-le-Soken"
}

ext["interchange"] = ext["station_name"].isin(interchanges)

# ----------------------------
# Country
# ----------------------------

ext["country"] = "England"

# ----------------------------
# Coastal stations
# ----------------------------

coastal = {
    "Burnham-on-Crouch",
    "Southminster",
    "Clacton-on-Sea",
    "Frinton-on-Sea",
    "Walton-on-the-Naze",
    "Harwich International",
    "Dovercourt",
    "Harwich Town"
}

ext["coastal"] = ext["station_name"].isin(coastal)

# ----------------------------
# County / Zone
# ----------------------------

ext["county"] = "Essex"

ext.loc[
    ext["route"] == "Sudbury Branch",
    "county"
] = "Suffolk"

ext["zone"] = ext["county"]

# ----------------------------
# Coordinates
# ----------------------------

lookup = stations.set_index("crsCode")

ext["latitude"] = ext["crs_code"].map(
    lookup["lat"]
)

ext["longitude"] = ext["crs_code"].map(
    lookup["long"]
)

# ----------------------------
# Liverpool Street coords
# ----------------------------

origin_lat = 51.517551
origin_lon = -0.080210

def haversine(lat1, lon1, lat2, lon2):

    R = 6371.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        sin(dlat/2)**2 +
        cos(lat1) *
        cos(lat2) *
        sin(dlon/2)**2
    )

    c = 2 * atan2(
        sqrt(a),
        sqrt(1-a)
    )

    return R * c

ext["distance_from_liverpool_street_km"] = ext.apply(
    lambda row: round(
        haversine(
            origin_lat,
            origin_lon,
            row["latitude"],
            row["longitude"]
        ),
        1
    ),
    axis=1
)

# ----------------------------
# Distance bands
# ----------------------------

def band(km):

    if km <= 20:
        return "local"
    elif km <= 80:
        return "regional"
    else:
        return "long"

ext["distance_band"] = (
    ext["distance_from_liverpool_street_km"]
    .apply(band)
)

# ----------------------------
# Approximate timings
# ----------------------------

timings = {
    "Billericay": 38,
    "Wickford": 42,
    "Battlesbridge": 47,
    "South Woodham Ferrers": 51,
    "North Fambridge": 56,
    "Althorne": 60,
    "Burnham-on-Crouch": 65,
    "Southminster": 71,
    "Rayleigh": 45,
    "Hockley": 49,
    "Rochford": 53,
    "Southend Airport": 56,
    "Prittlewell": 58,
    "Southend Victoria": 60,
    "White Notley": 58,
    "Cressing": 61,
    "Braintree Freeport": 64,
    "Braintree": 66,
    "Hythe": 70,
    "Wivenhoe": 74,
    "Alresford": 78,
    "Great Bentley": 82,
    "Weeley": 86,
    "Thorpe-le-Soken": 90,
    "Clacton-on-Sea": 96,
    "Kirby Cross": 95,
    "Frinton-on-Sea": 98,
    "Walton-on-the-Naze": 102,
    "Wrabness": 86,
    "Mistley": 82,
    "Harwich International": 98,
    "Dovercourt": 101,
    "Harwich Town": 104,
    "Chappel and Wakes Colne": 70,
    "Bures": 78,
    "Sudbury": 86
}

ext["liverpool_street_minutes"] = (
    ext["station_name"].map(timings)
)

# ----------------------------
# Save
# ----------------------------

ext.to_excel(
    "greateranglia_extensions_v1.xlsx",
    index=False
)

print("Saved: greateranglia_extensions_v1.xlsx")
print()
print("Stations:", len(ext))
print(
    "Missing coordinates:",
    ext["latitude"].isna().sum()
)
print(
    "Missing timings:",
    ext["liverpool_street_minutes"].isna().sum()
)

FileNotFoundError: [Errno 2] No such file or directory: 'stations.csv'

In [ ]:
import os

for f in sorted(os.listdir()):
    print(f)


.config
c2c_master.xlsx
east_anglia_master_v1_backup.xlsx
greateranglia_extensions.xlsx
greateranglia_master.xlsx
sample_data


In [ ]:
import pandas as pd

df = pd.read_excel("east_anglia_master_v1_backup.xlsx")

print("Stations:", len(df))
print()
print(df.columns.tolist())
print()
print(df.head())

Stations: 52

['station_id', 'station_name', 'crs', 'operator', 'route', 'origin_station', 'origin_minutes', 'distance_from_origin_km', 'terminus', 'interchange', 'latitude', 'longitude', 'country', 'county', 'zone', 'distance_band', 'coastal']

  station_id station_name  crs        operator                    route  \
0    c2c_020     Benfleet  BEN             c2c            Southend Line   
1    c2c_004      Barking  BKG             c2c            Southend Line   
2     ga_013    Brentwood  BRE  Greater Anglia  Great Eastern Main Line   
3    c2c_018     Basildon  BSO             c2c            Southend Line   
4     ga_016   Chelmsford  CFD  Greater Anglia  Great Eastern Main Line   

            origin_station  origin_minutes  distance_from_origin_km  terminus  \
0  London Fenchurch Street              40                    332.5     False   
1  London Fenchurch Street              12                     11.5     False   
2  London Liverpool Street              29                  

In [ ]:
import pandas as pd

df = pd.read_excel("east_anglia_master_v1_backup.xlsx")

print(
    df[
        [
            "station_name",
            "crs",
            "latitude",
            "longitude"
        ]
    ].sort_values("station_name")
    .head(20)
)


                 station_name  crs   latitude  longitude
1                     Barking  BKG  51.539982   0.080804
3                    Basildon  BSO  51.568672   0.457309
0                    Benfleet  BEN  54.115791  -2.510918
2                   Brentwood  BRE  51.613220   0.300830
6              Chadwell Heath  CHD  53.238209  -1.420110
5   Chafford Hundred Lakeside  CFH  51.502792   0.290938
7                   Chalkwell  CHW  51.538830   0.670600
4                  Chelmsford  CFD  53.724171  -1.355852
8                  Colchester  COL  51.900478   0.894085
9               Dagenham Dock  DAG  56.044178  -3.354809
10                       Diss  DIS  52.373096   1.123274
12               East Tilbury  ETL  51.485065   0.412479
11               Emerson Park  EMP  51.568890   0.220658
13                Forest Gate  FOG  51.549976   0.023568
15                 Gidea Park  GDP  51.581772   0.205411
16                  Goodmayes  GMY  51.565502   0.112277
17                      Grays  

In [ ]:
import pandas as pd

stations = pd.read_csv("stations.csv")

print(
    stations[
        stations["crsCode"] == "BEN"
    ][
        ["stationName", "crsCode", "lat", "long"]
    ]
)


    stationName crsCode        lat      long
217     Bentham     BEN  54.115791 -2.510918


In [ ]:
import pandas as pd

stations = pd.read_csv("stations.csv")

duplicates = (
    stations.groupby("crsCode")
    .size()
    .reset_index(name="count")
)

duplicates = duplicates[duplicates["count"] > 1]

print("Duplicate CRS codes:", len(duplicates))
print()
print(duplicates.head(20))


Duplicate CRS codes: 0

Empty DataFrame
Columns: [crsCode, count]
Index: []


In [ ]:
import pandas as pd

stations = pd.read_csv("stations.csv")

for station in [
    "Benfleet",
    "Chelmsford",
    "Dagenham Dock",
    "Chadwell Heath",
    "Romford",
    "Southend Central"
]:
    matches = stations[
        stations["stationName"]
        .str.contains(station, case=False, na=False)
    ]

    print("\n" + "="*50)
    print(station)
    print(matches[["stationName","crsCode","lat","long"]].head(20))


Benfleet
    stationName crsCode        lat      long
216    Benfleet     BEF  51.543964  0.561273

Chelmsford
    stationName crsCode        lat      long
505  Chelmsford     CHM  51.736595  0.469316

Dagenham Dock
       stationName crsCode        lat     long
651  Dagenham Dock     DDK  51.526237  0.14505

Chadwell Heath
        stationName crsCode        lat      long
483  Chadwell Heath     CTH  51.567905  0.128263

Romford
     stationName crsCode        lat      long
624     Cromford     CMF  53.112915 -1.548788
1958     Romford     RMF  51.575016  0.181986

Southend Central
           stationName crsCode        lat      long
2136  Southend Central     SOC  51.536972  0.712305


In [ ]:
import pandas as pd

master = pd.read_excel("east_anglia_master_v1_backup.xlsx")
stations = pd.read_csv("stations.csv")

station_lookup = (
    stations[
        ["stationName", "crsCode"]
    ]
    .drop_duplicates()
)

merged = master.merge(
    station_lookup,
    left_on="station_name",
    right_on="stationName",
    how="left"
)

mismatches = merged[
    merged["crs"] != merged["crsCode"]
]

print("Mismatches:", len(mismatches))
print()

print(
    mismatches[
        [
            "station_name",
            "crs",
            "crsCode"
        ]
    ].sort_values("station_name")
)

Mismatches: 9

                 station_name  crs crsCode
0                    Benfleet  BEN     BEF
6              Chadwell Heath  CHD     CTH
5   Chafford Hundred Lakeside  CFH     NaN
4                  Chelmsford  CFD     CHM
9               Dagenham Dock  DAG     DDK
35                     Pitsea  PIT     PSE
38                    Rainham  RNM     NaN
39           Stanford-le-Hope  SFO     NaN
43                  Stratford  SRA     NaN


In [ ]:
import pandas as pd

stations = pd.read_csv("stations.csv")

search_terms = [
    "Chafford",
    "Rainham",
    "Stanford",
    "Stratford"
]

for term in search_terms:

    print("\n" + "="*50)
    print(term)

    matches = stations[
        stations["stationName"]
        .str.contains(term, case=False, na=False)
    ]

    print(
        matches[
            ["stationName","crsCode","lat","long"]
        ].to_string(index=False)
    )



Chafford
     stationName crsCode       lat     long
Chafford Hundred     CFH 51.502792 0.290938

Rainham
    stationName crsCode       lat     long
Rainham (Essex)     RNM 51.517449 0.190716
 Rainham (Kent)     RAI 51.366570 0.611698

Stanford
     stationName crsCode       lat     long
Stanford-Le-Hope     SFO 51.513634 0.422634

Stratford
                stationName crsCode       lat      long
            Fenny Stratford     FEN 52.000103 -0.716636
         Stratford (London)     SRA 51.542358 -0.004175
    Stratford International     SFA 51.544306 -0.009882
        Stratford-upon-Avon     SAV 52.193760 -1.716148
Stratford-upon-Avon Parkway     STY 52.206420 -1.730700


In [ ]:
import pandas as pd

master = pd.read_excel("east_anglia_master_v1_backup.xlsx")
stations = pd.read_csv("stations.csv")

# --------------------------------------------------
# Fix CRS errors
# --------------------------------------------------

crs_fixes = {
    "Benfleet": "BEF",
    "Chadwell Heath": "CTH",
    "Chelmsford": "CHM",
    "Dagenham Dock": "DDK",
    "Pitsea": "PSE"
}

for station, crs in crs_fixes.items():

    master.loc[
        master["station_name"] == station,
        "crs"
    ] = crs

# --------------------------------------------------
# Name mappings
# --------------------------------------------------

name_map = {
    "Chafford Hundred Lakeside": "Chafford Hundred",
    "Rainham": "Rainham (Essex)",
    "Stanford-le-Hope": "Stanford-Le-Hope",
    "Stratford": "Stratford (London)"
}

# Create lookup name
master["lookup_name"] = master["station_name"]

for old, new in name_map.items():

    master.loc[
        master["station_name"] == old,
        "lookup_name"
    ] = new

# --------------------------------------------------
# Coordinate lookup by station name
# --------------------------------------------------

lookup = stations[
    [
        "stationName",
        "lat",
        "long"
    ]
].copy()

master = master.drop(
    columns=[
        "latitude",
        "longitude"
    ],
    errors="ignore"
)

master = master.merge(
    lookup,
    left_on="lookup_name",
    right_on="stationName",
    how="left"
)

master["latitude"] = master["lat"]
master["longitude"] = master["long"]

# Cleanup
master = master.drop(
    columns=[
        "lookup_name",
        "stationName",
        "lat",
        "long"
    ]
)

# --------------------------------------------------
# Save
# --------------------------------------------------

master.to_excel(
    "east_anglia_master_v2.xlsx",
    index=False
)

print("Saved: east_anglia_master_v2.xlsx")
print()

print(
    "Missing coordinates:",
    master["latitude"].isna().sum()
)

print()

print(
    master[
        master["station_name"].isin([
            "Benfleet",
            "Chelmsford",
            "Dagenham Dock",
            "Pitsea",
            "Stratford"
        ])
    ][
        [
            "station_name",
            "crs",
            "latitude",
            "longitude"
        ]
    ]
)

Saved: east_anglia_master_v2.xlsx

Missing coordinates: 0

     station_name  crs   latitude  longitude
0        Benfleet  BEF  51.543964   0.561273
4      Chelmsford  CHM  51.736595   0.469316
9   Dagenham Dock  DDK  51.526237   0.145050
35         Pitsea  PSE  51.560387   0.508806
43      Stratford  SRA  51.542358  -0.004175


In [ ]:
import pandas as pd
from math import radians, sin, cos, sqrt, atan2

master = pd.read_excel("east_anglia_master_v2.xlsx")

# ----------------------------
# Origins
# ----------------------------

origins = {
    "c2c": (
        51.511234,   # Fenchurch Street
        -0.079039
    ),
    "Greater Anglia": (
        51.517551,   # Liverpool Street
        -0.080210
    )
}

# ----------------------------
# Haversine
# ----------------------------

def haversine(lat1, lon1, lat2, lon2):

    R = 6371.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        sin(dlat / 2) ** 2
        + cos(lat1)
        * cos(lat2)
        * sin(dlon / 2) ** 2
    )

    c = 2 * atan2(
        sqrt(a),
        sqrt(1 - a)
    )

    return R * c

# ----------------------------
# Recalculate distances
# ----------------------------

distances = []

for _, row in master.iterrows():

    origin_lat, origin_lon = origins[row["operator"]]

    km = haversine(
        origin_lat,
        origin_lon,
        row["latitude"],
        row["longitude"]
    )

    distances.append(round(km, 1))

master["distance_from_origin_km"] = distances

# ----------------------------
# Rebuild distance bands
# ----------------------------

def band(km):

    if km <= 20:
        return "local"
    elif km <= 80:
        return "regional"
    else:
        return "long"

master["distance_band"] = (
    master["distance_from_origin_km"]
    .apply(band)
)

# ----------------------------
# Save
# ----------------------------

master.to_excel(
    "east_anglia_master_v3.xlsx",
    index=False
)

print("Saved: east_anglia_master_v3.xlsx")
print()

print(
    master[
        [
            "station_name",
            "operator",
            "distance_from_origin_km",
            "distance_band"
        ]
    ]
    .sort_values("distance_from_origin_km")
    .head(15)
)

In [ ]:
import pandas as pd
from math import radians, sin, cos, sqrt, atan2

master = pd.read_excel("east_anglia_master_v2.xlsx")

# ----------------------------
# Origins
# ----------------------------

origins = {
    "c2c": (
        51.511234,   # Fenchurch Street
        -0.079039
    ),
    "Greater Anglia": (
        51.517551,   # Liverpool Street
        -0.080210
    )
}

# ----------------------------
# Haversine
# ----------------------------

def haversine(lat1, lon1, lat2, lon2):

    R = 6371.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        sin(dlat / 2) ** 2
        + cos(lat1)
        * cos(lat2)
        * sin(dlon / 2) ** 2
    )

    c = 2 * atan2(
        sqrt(a),
        sqrt(1 - a)
    )

    return R * c

# ----------------------------
# Recalculate distances
# ----------------------------

distances = []

for _, row in master.iterrows():

    origin_lat, origin_lon = origins[row["operator"]]

    km = haversine(
        origin_lat,
        origin_lon,
        row["latitude"],
        row["longitude"]
    )

    distances.append(round(km, 1))

master["distance_from_origin_km"] = distances

# ----------------------------
# Rebuild distance bands
# ----------------------------

def band(km):

    if km <= 20:
        return "local"
    elif km <= 80:
        return "regional"
    else:
        return "long"

master["distance_band"] = (
    master["distance_from_origin_km"]
    .apply(band)
)

# ----------------------------
# Save
# ----------------------------

master.to_excel(
    "east_anglia_master_v3.xlsx",
    index=False
)

print("Saved: east_anglia_master_v3.xlsx")
print()

print(
    master[
        [
            "station_name",
            "operator",
            "distance_from_origin_km",
            "distance_band"
        ]
    ]
    .sort_values("distance_from_origin_km")
    .head(15)
)

Saved: east_anglia_master_v3.xlsx

               station_name        operator  distance_from_origin_km  \
14  London Fenchurch Street             c2c                      0.0   
27  London Liverpool Street  Greater Anglia                      0.0   
26                Limehouse             c2c                      2.7   
43                Stratford  Greater Anglia                      5.9   
50                 West Ham             c2c                      6.2   
31                 Maryland  Greater Anglia                      6.7   
13              Forest Gate  Greater Anglia                      8.0   
30               Manor Park  Greater Anglia                      9.5   
20                   Ilford  Greater Anglia                     11.3   
1                   Barking             c2c                     11.5   
45              Seven Kings  Greater Anglia                     13.3   
16                Goodmayes  Greater Anglia                     14.3   
6            Chadwell Heath  

In [ ]:
import pandas as pd

master = pd.read_excel("east_anglia_master_v3.xlsx")

print(
    master[
        master["station_name"].isin([
            "Benfleet",
            "Basildon",
            "Southend Central",
            "Chelmsford",
            "Colchester",
            "Ipswich",
            "Norwich"
        ])
    ][
        [
            "station_name",
            "operator",
            "distance_from_origin_km",
            "distance_band"
        ]
    ]
    .sort_values("distance_from_origin_km")
)

        station_name        operator  distance_from_origin_km distance_band
3           Basildon             c2c                     37.6      regional
0           Benfleet             c2c                     44.4      regional
4         Chelmsford  Greater Anglia                     45.1      regional
42  Southend Central             c2c                     54.8      regional
8         Colchester  Greater Anglia                     79.5      regional
22           Ipswich  Greater Anglia                    103.0          long
32           Norwich  Greater Anglia                    155.6          long


In [ ]:
import pandas as pd

ext = pd.read_excel("greateranglia_extensions.xlsx")
stations = pd.read_csv("stations.csv")

station_lookup = stations[
    ["stationName", "crsCode"]
].drop_duplicates()

merged = ext.merge(
    station_lookup,
    left_on="station_name",
    right_on="stationName",
    how="left"
)

mismatches = merged[
    merged["crs_code"] != merged["crsCode"]
]

print("Total stations:", len(ext))
print("CRS mismatches:", len(mismatches))
print()

print(
    mismatches[
        [
            "station_name",
            "crs_code",
            "crsCode"
        ]
    ].sort_values("station_name")
)

Total stations: 36
CRS mismatches: 15

               station_name crs_code crsCode
20                Alresford      AIR     ALR
5                  Althorne      AHO     ALN
2             Battlesbridge      BSB     BLB
34                    Bures      BUR     BUE
6         Burnham-on-Crouch      BUC     NaN
33  Chappel and Wakes Colne      CWC     NaN
24           Clacton-on-Sea      CLT     NaN
15                 Cressing      CRS     CES
26           Frinton-on-Sea      FRI     NaN
30    Harwich International      HWC     HPQ
32             Harwich Town      HWT     HWC
7              Southminster      SMS     SMN
35                  Sudbury      SUY     NaN
23          Thorpe-le-Soken      TLS     NaN
27       Walton-on-the-Naze      WON     NaN


In [ ]:
import pandas as pd

ext = pd.read_excel("greateranglia_extensions.xlsx")

crs_fixes = {
    "Alresford": "ALR",
    "Althorne": "ALN",
    "Battlesbridge": "BLB",
    "Bures": "BUE",
    "Cressing": "CES",
    "Harwich International": "HPQ",
    "Harwich Town": "HWC",
    "Southminster": "SMN"
}

for station, crs in crs_fixes.items():

    ext.loc[
        ext["station_name"] == station,
        "crs_code"
    ] = crs

ext.to_excel(
    "greateranglia_extensions_fixed.xlsx",
    index=False
)

print("CRS corrections applied")

CRS corrections applied


In [ ]:
import pandas as pd

stations = pd.read_csv("stations.csv")

search_terms = [
    "Burnham",
    "Chappel",
    "Clacton",
    "Frinton",
    "Sudbury",
    "Thorpe",
    "Walton"
]

for term in search_terms:

    print("\n" + "=" * 50)
    print(term)

    matches = stations[
        stations["stationName"]
        .str.contains(term, case=False, na=False)
    ]

    print(
        matches[
            ["stationName", "crsCode"]
        ].to_string(index=False)
    )


Burnham
         stationName crsCode
             Burnham     BNM
   Burnham-On-Crouch     BUU
Highbridge & Burnham     HIG

Chappel
          stationName crsCode
Chappel & Wakes Colne     CWC

Clacton
   stationName crsCode
Clacton-On-Sea     CLT

Frinton
   stationName crsCode
Frinton-On-Sea     FRI

Sudbury
          stationName crsCode
Sudbury & Harrow Road     SUD
  Sudbury Hill Harrow     SDH
      Sudbury Suffolk     SUY

Thorpe
    stationName crsCode
       Althorpe     ALP
    Cleethorpes     CLE
     Goldthorpe     GOE
     Moorthorpe     MRP
      Nunthorpe     NNT
   Ravensthorpe     RVN
     Scunthorpe     SCU
     Thorpe Bay     TPB
 Thorpe Culvert     TPC
Thorpe-Le-Soken     TLS

Walton
        stationName crsCode
Walton (Merseyside)     WAO
   Walton-On-Thames     WAL
 Walton-On-The-Naze     WON


In [ ]:
import pandas as pd
from math import radians, sin, cos, sqrt, atan2

# ==================================================
# LOAD FILES
# ==================================================

ext = pd.read_excel("greateranglia_extensions_fixed.xlsx")
stations = pd.read_csv("stations.csv")

# ==================================================
# FINAL CRS FIXES
# ==================================================

extra_fixes = {
    "Burnham-on-Crouch": "BUU",
    "Chappel and Wakes Colne": "CWC",
    "Clacton-on-Sea": "CLT",
    "Frinton-on-Sea": "FRI",
    "Sudbury": "SUY",
    "Thorpe-le-Soken": "TLS",
    "Walton-on-the-Naze": "WON"
}

for station, crs in extra_fixes.items():

    ext.loc[
        ext["station_name"] == station,
        "crs_code"
    ] = crs

# ==================================================
# STATION IDS
# ==================================================

ext["station_id"] = [
    f"ga_{str(i+27).zfill(3)}"
    for i in range(len(ext))
]

# ==================================================
# ROUTES
# ==================================================

route_map = {

    "Southend Victoria Line": [
        "Billericay","Wickford","Rayleigh",
        "Hockley","Rochford",
        "Southend Airport",
        "Prittlewell",
        "Southend Victoria"
    ],

    "Southminster Branch": [
        "Battlesbridge",
        "South Woodham Ferrers",
        "North Fambridge",
        "Althorne",
        "Burnham-on-Crouch",
        "Southminster"
    ],

    "Braintree Branch": [
        "White Notley",
        "Cressing",
        "Braintree Freeport",
        "Braintree"
    ],

    "Clacton/Walton Branch": [
        "Hythe",
        "Wivenhoe",
        "Alresford",
        "Great Bentley",
        "Weeley",
        "Thorpe-le-Soken",
        "Clacton-on-Sea",
        "Kirby Cross",
        "Frinton-on-Sea",
        "Walton-on-the-Naze"
    ],

    "Harwich Branch": [
        "Wrabness",
        "Mistley",
        "Harwich International",
        "Dovercourt",
        "Harwich Town"
    ],

    "Sudbury Branch": [
        "Chappel and Wakes Colne",
        "Bures",
        "Sudbury"
    ]
}

def get_route(station):

    for route, stations_list in route_map.items():

        if station in stations_list:
            return route

    return "Unknown"

ext["route"] = ext["station_name"].apply(get_route)

# ==================================================
# FLAGS
# ==================================================

termini = {
    "Southend Victoria",
    "Southminster",
    "Braintree",
    "Clacton-on-Sea",
    "Walton-on-the-Naze",
    "Harwich Town",
    "Sudbury"
}

ext["terminus"] = ext["station_name"].isin(termini)

interchanges = {
    "Wickford",
    "Thorpe-le-Soken"
}

ext["interchange"] = ext["station_name"].isin(interchanges)

ext["country"] = "England"

coastal = {
    "Burnham-on-Crouch",
    "Southminster",
    "Clacton-on-Sea",
    "Frinton-on-Sea",
    "Walton-on-the-Naze",
    "Harwich International",
    "Dovercourt",
    "Harwich Town"
}

ext["coastal"] = ext["station_name"].isin(coastal)

# ==================================================
# COUNTY / ZONE
# ==================================================

ext["county"] = "Essex"

ext.loc[
    ext["route"] == "Sudbury Branch",
    "county"
] = "Suffolk"

ext["zone"] = ext["county"]

# ==================================================
# NAME MAPPINGS
# ==================================================

name_map = {
    "Burnham-on-Crouch": "Burnham-On-Crouch",
    "Chappel and Wakes Colne": "Chappel & Wakes Colne",
    "Clacton-on-Sea": "Clacton-On-Sea",
    "Frinton-on-Sea": "Frinton-On-Sea",
    "Sudbury": "Sudbury Suffolk",
    "Thorpe-le-Soken": "Thorpe-Le-Soken",
    "Walton-on-the-Naze": "Walton-On-The-Naze"
}

ext["lookup_name"] = ext["station_name"]

for old, new in name_map.items():

    ext.loc[
        ext["station_name"] == old,
        "lookup_name"
    ] = new

# ==================================================
# COORDINATES
# ==================================================

lookup = stations[
    [
        "stationName",
        "lat",
        "long"
    ]
]

ext = ext.merge(
    lookup,
    left_on="lookup_name",
    right_on="stationName",
    how="left"
)

ext["latitude"] = ext["lat"]
ext["longitude"] = ext["long"]

# ==================================================
# DISTANCES
# ==================================================

origin_lat = 51.517551
origin_lon = -0.080210

def haversine(lat1, lon1, lat2, lon2):

    R = 6371.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        sin(dlat/2)**2
        + cos(lat1)
        * cos(lat2)
        * sin(dlon/2)**2
    )

    c = 2 * atan2(
        sqrt(a),
        sqrt(1-a)
    )

    return R * c

ext["distance_from_origin_km"] = ext.apply(
    lambda r: round(
        haversine(
            origin_lat,
            origin_lon,
            r["latitude"],
            r["longitude"]
        ),
        1
    ),
    axis=1
)

# ==================================================
# BANDS
# ==================================================

def band(km):

    if km <= 20:
        return "local"
    elif km <= 80:
        return "regional"
    return "long"

ext["distance_band"] = (
    ext["distance_from_origin_km"]
    .apply(band)
)

# ==================================================
# ORIGIN
# ==================================================

ext["origin_station"] = "London Liverpool Street"

# ==================================================
# SAVE
# ==================================================

ext.to_excel(
    "greateranglia_extensions_v1.xlsx",
    index=False
)

print("Saved: greateranglia_extensions_v1.xlsx")
print()
print("Stations:", len(ext))
print("Missing coordinates:", ext["latitude"].isna().sum())
print("Unknown routes:", (ext["route"]=="Unknown").sum())

Saved: greateranglia_extensions_v1.xlsx

Stations: 36
Missing coordinates: 0
Unknown routes: 0


In [ ]:
import pandas as pd

master = pd.read_excel("east_anglia_master_v3.xlsx")
ext = pd.read_excel("greateranglia_extensions_v1.xlsx")

# Keep only the master schema columns
master_columns = master.columns.tolist()

ext = ext.reindex(columns=master_columns)

east_anglia_v4 = pd.concat(
    [master, ext],
    ignore_index=True
)

east_anglia_v4.to_excel(
    "east_anglia_master_v4.xlsx",
    index=False
)

east_anglia_v4.to_csv(
    "east_anglia_master_v4.csv",
    index=False
)

print("Stations:", len(east_anglia_v4))
print()

print("By operator:")
print(
    east_anglia_v4["operator"]
    .value_counts()
)

print()
print("By route:")
print(
    east_anglia_v4["route"]
    .value_counts()
)

print()
print("Missing values:")
print(
    east_anglia_v4.isnull().sum()[
        east_anglia_v4.isnull().sum() > 0
    ]
)

Stations: 88

By operator:
operator
Greater Anglia    62
c2c               26
Name: count, dtype: int64

By route:
route
Great Eastern Main Line    26
Southend Line              18
Clacton/Walton Branch      10
Southend Victoria Line      8
Southminster Branch         6
Harwich Branch              5
Braintree Branch            4
Tilbury Loop                4
Sudbury Branch              3
Ockendon Branch             2
Romford Branch              2
Name: count, dtype: int64

Missing values:
crs               36
origin_minutes    36
dtype: int64


In [ ]:
ext = pd.read_excel("greateranglia_extensions_v1.xlsx")

print(ext.columns.tolist())

['station_name', 'crs_code', 'operator', 'station_id', 'route', 'terminus', 'interchange', 'country', 'coastal', 'county', 'zone', 'lookup_name', 'stationName', 'lat', 'long', 'latitude', 'longitude', 'distance_from_origin_km', 'distance_band', 'origin_station']


In [ ]:
import pandas as pd

ext = pd.read_excel("greateranglia_extensions_v1.xlsx")

# --------------------------------------------------
# CRS
# --------------------------------------------------

ext["crs"] = ext["crs_code"]

# --------------------------------------------------
# Liverpool Street timings
# --------------------------------------------------

timings = {
    "Billericay": 38,
    "Wickford": 42,
    "Battlesbridge": 47,
    "South Woodham Ferrers": 51,
    "North Fambridge": 56,
    "Althorne": 60,
    "Burnham-on-Crouch": 65,
    "Southminster": 71,
    "Rayleigh": 45,
    "Hockley": 49,
    "Rochford": 53,
    "Southend Airport": 56,
    "Prittlewell": 58,
    "Southend Victoria": 60,
    "White Notley": 58,
    "Cressing": 61,
    "Braintree Freeport": 64,
    "Braintree": 66,
    "Hythe": 70,
    "Wivenhoe": 74,
    "Alresford": 78,
    "Great Bentley": 82,
    "Weeley": 86,
    "Thorpe-le-Soken": 90,
    "Clacton-on-Sea": 96,
    "Kirby Cross": 95,
    "Frinton-on-Sea": 98,
    "Walton-on-the-Naze": 102,
    "Wrabness": 86,
    "Mistley": 82,
    "Harwich International": 98,
    "Dovercourt": 101,
    "Harwich Town": 104,
    "Chappel and Wakes Colne": 70,
    "Bures": 78,
    "Sudbury": 86
}

ext["origin_minutes"] = ext["station_name"].map(timings)

# --------------------------------------------------
# Remove temporary columns
# --------------------------------------------------

drop_cols = [
    "lookup_name",
    "stationName",
    "lat",
    "long"
]

for col in drop_cols:
    if col in ext.columns:
        ext = ext.drop(columns=col)

# --------------------------------------------------
# Save
# --------------------------------------------------

ext.to_excel(
    "greateranglia_extensions_v2.xlsx",
    index=False
)

print("Saved: greateranglia_extensions_v2.xlsx")
print()

print("Missing CRS:", ext["crs"].isna().sum())
print("Missing timings:", ext["origin_minutes"].isna().sum())

print()
print(ext.columns.tolist())

Saved: greateranglia_extensions_v2.xlsx

Missing CRS: 0
Missing timings: 0

['station_name', 'crs_code', 'operator', 'station_id', 'route', 'terminus', 'interchange', 'country', 'coastal', 'county', 'zone', 'latitude', 'longitude', 'distance_from_origin_km', 'distance_band', 'origin_station', 'crs', 'origin_minutes']


In [ ]:
import pandas as pd

master = pd.read_excel("east_anglia_master_v3.xlsx")
ext = pd.read_excel("greateranglia_extensions_v2.xlsx")

# Match master schema exactly
ext = ext.reindex(columns=master.columns)

east_anglia_v5 = pd.concat(
    [master, ext],
    ignore_index=True
)

east_anglia_v5.to_excel(
    "east_anglia_master_v5.xlsx",
    index=False
)

east_anglia_v5.to_csv(
    "east_anglia_master_v5.csv",
    index=False
)

print("Stations:", len(east_anglia_v5))
print()

print("By operator:")
print(east_anglia_v5["operator"].value_counts())

print()
print("By route:")
print(east_anglia_v5["route"].value_counts())

print()
print("Missing values:")
missing = east_anglia_v5.isnull().sum()
print(missing[missing > 0])

print()
print("Duplicate station names:",
      east_anglia_v5["station_name"].duplicated().sum())

print("Duplicate CRS:",
      east_anglia_v5["crs"].duplicated().sum())

Stations: 88

By operator:
operator
Greater Anglia    62
c2c               26
Name: count, dtype: int64

By route:
route
Great Eastern Main Line    26
Southend Line              18
Clacton/Walton Branch      10
Southend Victoria Line      8
Southminster Branch         6
Harwich Branch              5
Braintree Branch            4
Tilbury Loop                4
Sudbury Branch              3
Ockendon Branch             2
Romford Branch              2
Name: count, dtype: int64

Missing values:
Series([], dtype: int64)

Duplicate station names: 1
Duplicate CRS: 1


In [ ]:
import shutil

shutil.copy(
    "east_anglia_master_v5.xlsx",
    "east_anglia_master_v5_RELEASE.xlsx"
)

shutil.copy(
    "east_anglia_master_v5.csv",
    "east_anglia_master_v5_RELEASE.csv"
)

print("Release copies created")

Release copies created


In [ ]:
import pandas as pd

# ==========================================
# LOAD
# ==========================================

df = pd.read_excel(
    "east_anglia_master_v5_RELEASE.xlsx"
)

# ==========================================
# NORTHING / EASTING RANKS
# ==========================================

df["northing_rank"] = (
    df["latitude"]
    .rank(method="dense")
    .astype(int)
)

df["easting_rank"] = (
    df["longitude"]
    .rank(method="dense")
    .astype(int)
)

# ==========================================
# SEASIDE RESORTS
# ==========================================

seaside = {
    "Southend Central",
    "Westcliff",
    "Chalkwell",
    "Leigh-on-Sea",
    "Thorpe Bay",
    "Shoeburyness",
    "Clacton-on-Sea",
    "Frinton-on-Sea",
    "Walton-on-the-Naze",
    "Harwich Town",
    "Dovercourt"
}

df["seaside_resort"] = (
    df["station_name"]
    .isin(seaside)
)

# ==========================================
# AIRPORT STATIONS
# ==========================================

airport = {
    "Southend Airport"
}

df["airport_station"] = (
    df["station_name"]
    .isin(airport)
)

# ==========================================
# BRANCH JUNCTIONS
# ==========================================

junctions = {
    "Wickford",
    "Thorpe-le-Soken",
    "Marks Tey",
    "Manningtree",
    "Witham",
    "Romford"
}

df["branch_junction"] = (
    df["station_name"]
    .isin(junctions)
)

# ==========================================
# MAJOR INTERCHANGES
# ==========================================

major = {
    "London Liverpool Street",
    "Stratford",
    "Romford",
    "Barking",
    "West Ham",
    "Shenfield",
    "Colchester",
    "Ipswich",
    "Norwich"
}

df["major_interchange"] = (
    df["station_name"]
    .isin(major)
)

# ==========================================
# TERMINUS TYPE
# ==========================================

def terminus_type(row):

    if not row["terminus"]:
        return "None"

    if row["station_name"] in {
        "London Liverpool Street",
        "London Fenchurch Street"
    }:
        return "London"

    if row["coastal"]:
        return "Coastal"

    return "Branch"

df["terminus_type"] = df.apply(
    terminus_type,
    axis=1
)

# ==========================================
# SAVE
# ==========================================

df.to_excel(
    "east_anglia_master_v6.xlsx",
    index=False
)

print("Saved: east_anglia_master_v6.xlsx")
print()

print("Stations:", len(df))

print()
print("Seaside stations:",
      df["seaside_resort"].sum())

print("Airport stations:",
      df["airport_station"].sum())

print("Branch junctions:",
      df["branch_junction"].sum())

print("Major interchanges:",
      df["major_interchange"].sum())

Saved: east_anglia_master_v6.xlsx

Stations: 88

Seaside stations: 11
Airport stations: 1
Branch junctions: 7
Major interchanges: 10


In [ ]:
import pandas as pd

landmarks = [

    # Essex
    ["Southend Pier","Pier","Essex"],
    ["Adventure Island","Theme Park","Essex"],
    ["Southend Cliff Gardens","Historic Site","Essex"],
    ["Hadleigh Castle","Castle","Essex"],
    ["RHS Hyde Hall","Nature Reserve","Essex"],
    ["Chelmsford Cathedral","Cathedral","Essex"],
    ["Hylands House","Country House","Essex"],
    ["Layer Marney Tower","Historic Site","Essex"],
    ["Audley End House","Country House","Essex"],
    ["Colchester Castle","Castle","Essex"],
    ["Colchester Zoo","Nature Reserve","Essex"],
    ["Beth Chatto Gardens","Nature Reserve","Essex"],
    ["Firstsite","Museum","Essex"],
    ["Jaywick Beach","Beach","Essex"],
    ["Walton Pier","Pier","Essex"],
    ["Clacton Pier","Pier","Essex"],
    ["Frinton Greensward","Historic Site","Essex"],
    ["Harwich Redoubt Fort","Historic Site","Essex"],

    # Suffolk
    ["Sutton Hoo","Historic Site","Suffolk"],
    ["Framlingham Castle","Castle","Suffolk"],
    ["Orford Castle","Castle","Suffolk"],
    ["Aldeburgh Moot Hall","Museum","Suffolk"],
    ["Southwold Pier","Pier","Suffolk"],
    ["Snape Maltings","Historic Site","Suffolk"],
    ["Latitude Festival Site","Festival Site","Suffolk"],
    ["Christchurch Mansion","Museum","Suffolk"],
    ["Ipswich Waterfront","Historic Site","Suffolk"],
    ["Flatford Mill","Historic Site","Suffolk"],
    ["Minsmere Reserve","Nature Reserve","Suffolk"],
    ["Bury St Edmunds Abbey","Historic Site","Suffolk"],

    # Norfolk
    ["Norwich Cathedral","Cathedral","Norfolk"],
    ["Norwich Castle","Castle","Norfolk"],
    ["Sandringham House","Country House","Norfolk"],
    ["Holkham Hall","Country House","Norfolk"],
    ["Blickling Hall","Country House","Norfolk"],
    ["Great Yarmouth Pleasure Beach","Theme Park","Norfolk"],
    ["Time and Tide Museum","Museum","Norfolk"],
    ["Cromer Pier","Pier","Norfolk"],
    ["Sheringham Park","Nature Reserve","Norfolk"],
    ["Norfolk Broads","Nature Reserve","Norfolk"],
    ["Sainsbury Centre","Museum","Norfolk"],
    ["University of East Anglia","University","Norfolk"],
    ["Burgh Castle","Castle","Norfolk"],
    ["Wymondham Abbey","Abbey","Norfolk"],
    ["The Forum Norwich","Museum","Norfolk"],

    # London
    ["Tower of London","Historic Site","London"],
    ["St Paul's Cathedral","Cathedral","London"],
    ["Canary Wharf","Landmark","London"],
    ["Greenwich Observatory","Historic Site","London"],
    ["O2 Arena","Sports Venue","London"]

]

df = pd.DataFrame(
    landmarks,
    columns=[
        "landmark_name",
        "category",
        "county"
    ]
)

df["landmark_id"] = [
    f"lm_{str(i+1).zfill(3)}"
    for i in range(len(df))
]

# reorder columns

df = df[
    [
        "landmark_id",
        "landmark_name",
        "category",
        "county"
    ]
]

df.to_excel(
    "east_anglia_landmarks_v1.xlsx",
    index=False
)

print("Saved: east_anglia_landmarks_v1.xlsx")
print("Landmarks:", len(df))

print()
print(df.head())

Saved: east_anglia_landmarks_v1.xlsx
Landmarks: 50

  landmark_id           landmark_name        category county
0      lm_001           Southend Pier            Pier  Essex
1      lm_002        Adventure Island      Theme Park  Essex
2      lm_003  Southend Cliff Gardens   Historic Site  Essex
3      lm_004         Hadleigh Castle          Castle  Essex
4      lm_005           RHS Hyde Hall  Nature Reserve  Essex


In [ ]:
import pandas as pd

lm = pd.read_excel("east_anglia_landmarks_v1.xlsx")

lm["latitude"] = None
lm["longitude"] = None

lm.to_excel(
    "east_anglia_landmarks_v2.xlsx",
    index=False
)

print("Saved: east_anglia_landmarks_v2.xlsx")
print("Landmarks:", len(lm))
print(lm.head())

Saved: east_anglia_landmarks_v2.xlsx
Landmarks: 50
  landmark_id           landmark_name        category county latitude  \
0      lm_001           Southend Pier            Pier  Essex     None   
1      lm_002        Adventure Island      Theme Park  Essex     None   
2      lm_003  Southend Cliff Gardens   Historic Site  Essex     None   
3      lm_004         Hadleigh Castle          Castle  Essex     None   
4      lm_005           RHS Hyde Hall  Nature Reserve  Essex     None   

  longitude  
0      None  
1      None  
2      None  
3      None  
4      None  


In [ ]:
import pandas as pd
import requests
import time

# Load landmarks
lm = pd.read_excel("east_anglia_landmarks_v2.xlsx")

# Geocode function
def geocode(place):

    try:

        url = "https://nominatim.openstreetmap.org/search"

        params = {
            "q": place,
            "format": "json",
            "limit": 1
        }

        r = requests.get(
            url,
            params=params,
            headers={
                "User-Agent": "TheRouteLandmarkBuilder"
            },
            timeout=20
        )

        results = r.json()

        if len(results) == 0:
            return None, None

        return (
            float(results[0]["lat"]),
            float(results[0]["lon"])
        )

    except Exception:
        return None, None

# Geocode landmarks
for idx, row in lm.iterrows():

    query = f"{row['landmark_name']}, {row['county']}, England"

    lat, lon = geocode(query)

    lm.at[idx, "latitude"] = lat
    lm.at[idx, "longitude"] = lon

    print(
        f"{idx+1:02d}/50",
        row["landmark_name"],
        lat,
        lon
    )

    time.sleep(1)

# Save
lm.to_excel(
    "east_anglia_landmarks_v3.xlsx",
    index=False
)

print()
print("Saved: east_anglia_landmarks_v3.xlsx")

print(
    "Missing coordinates:",
    lm["latitude"].isna().sum()
)


01/50 Southend Pier 51.5240699 0.7187458
02/50 Adventure Island 51.5327119 0.717746
03/50 Southend Cliff Gardens None None
04/50 Hadleigh Castle 51.5445746 0.6087095
05/50 RHS Hyde Hall None None
06/50 Chelmsford Cathedral 51.7352492 0.4723482
07/50 Hylands House 51.711178 0.437246
08/50 Layer Marney Tower 51.8227188 0.7970269
09/50 Audley End House 52.0208569 0.22062
10/50 Colchester Castle 51.8906379 0.9030675
11/50 Colchester Zoo 51.8607677 0.8313362
12/50 Beth Chatto Gardens 51.8745543 1.0042029
13/50 Firstsite 51.8888945 0.9057202
14/50 Jaywick Beach 51.7740495 1.1194701
15/50 Walton Pier 51.8463105 1.2723964
16/50 Clacton Pier 51.7854578 1.1558952
17/50 Frinton Greensward None None
18/50 Harwich Redoubt Fort 51.9421502 1.2879533
19/50 Sutton Hoo 52.0893526 1.3386941
20/50 Framlingham Castle 52.2242205 1.3466796
21/50 Orford Castle 52.0942831 1.5307271
22/50 Aldeburgh Moot Hall None None
23/50 Southwold Pier 52.3310958 1.6753157
24/50 Snape Maltings 52.1634227 1.4967704
25/50 Lati

In [ ]:
import pandas as pd
import requests
import time

lm = pd.read_excel("east_anglia_landmarks_v3.xlsx")

manual_queries = {
    "Southend Cliff Gardens": "Cliff Gardens Southend-on-Sea Essex",
    "RHS Hyde Hall": "RHS Garden Hyde Hall Essex",
    "Frinton Greensward": "The Greensward Frinton-on-Sea Essex",
    "Aldeburgh Moot Hall": "Moot Hall Aldeburgh Suffolk",
    "Latitude Festival Site": "Henham Park Suffolk",
    "Minsmere Reserve": "RSPB Minsmere Suffolk",
    "Great Yarmouth Pleasure Beach": "Pleasure Beach Great Yarmouth Norfolk",
    "Time and Tide Museum": "Time and Tide Museum Great Yarmouth"
}

def geocode(query):

    try:
        r = requests.get(
            "https://nominatim.openstreetmap.org/search",
            params={
                "q": query,
                "format": "json",
                "limit": 1
            },
            headers={
                "User-Agent": "TheRouteLandmarks"
            },
            timeout=20
        )

        data = r.json()

        if len(data) == 0:
            return None, None

        return (
            float(data[0]["lat"]),
            float(data[0]["lon"])
        )

    except:
        return None, None

for landmark, query in manual_queries.items():

    idx = lm.index[
        lm["landmark_name"] == landmark
    ][0]

    lat, lon = geocode(query)

    lm.loc[idx, "latitude"] = lat
    lm.loc[idx, "longitude"] = lon

    print(
        landmark,
        lat,
        lon
    )

    time.sleep(1)

lm.to_excel(
    "east_anglia_landmarks_v4.xlsx",
    index=False
)

print()
print(
    "Remaining missing:",
    lm["latitude"].isna().sum()
)

Southend Cliff Gardens 51.5392729 0.6693724
RHS Hyde Hall None None
Frinton Greensward 51.8301264 1.2484626
Aldeburgh Moot Hall 52.1549158 1.6029416
Latitude Festival Site None None
Minsmere Reserve 52.2473915 1.6195559
Great Yarmouth Pleasure Beach 52.5932831 1.7359856
Time and Tide Museum None None

Remaining missing: 3


In [ ]:
import pandas as pd
import requests
import time

lm = pd.read_excel("east_anglia_landmarks_v4.xlsx")

manual_queries = {
    "RHS Hyde Hall": "RHS Garden Hyde Hall, Rettendon, Essex",
    "Latitude Festival Site": "Henham Park, Beccles, Suffolk",
    "Time and Tide Museum": "Time and Tide Museum of Great Yarmouth Life"
}

def geocode(query):

    try:

        r = requests.get(
            "https://nominatim.openstreetmap.org/search",
            params={
                "q": query,
                "format": "json",
                "limit": 1
            },
            headers={
                "User-Agent": "TheRouteLandmarks"
            },
            timeout=20
        )

        data = r.json()

        if len(data) == 0:
            return None, None

        return (
            float(data[0]["lat"]),
            float(data[0]["lon"])
        )

    except:
        return None, None

for landmark, query in manual_queries.items():

    idx = lm.index[
        lm["landmark_name"] == landmark
    ][0]

    lat, lon = geocode(query)

    lm.loc[idx, "latitude"] = lat
    lm.loc[idx, "longitude"] = lon

    print(landmark, lat, lon)

    time.sleep(1)

lm.to_excel(
    "east_anglia_landmarks_v5.xlsx",
    index=False
)

print()
print(
    "Remaining missing:",
    lm["latitude"].isna().sum()
)

RHS Hyde Hall None None
Latitude Festival Site None None
Time and Tide Museum None None

Remaining missing: 3


In [ ]:
import pandas as pd

lm = pd.read_excel("east_anglia_landmarks_v4.xlsx")

manual_coords = {

    "RHS Hyde Hall": (
        51.6766,
        0.4867
    ),

    "Latitude Festival Site": (
        52.3358,
        1.5689
    ),

    "Time and Tide Museum": (
        52.6078,
        1.7334
    )
}

for landmark, (lat, lon) in manual_coords.items():

    idx = lm.index[
        lm["landmark_name"] == landmark
    ][0]

    lm.loc[idx, "latitude"] = lat
    lm.loc[idx, "longitude"] = lon

lm.to_excel(
    "east_anglia_landmarks_v5.xlsx",
    index=False
)

print(
    "Remaining missing:",
    lm["latitude"].isna().sum()
)

Remaining missing: 0


In [ ]:
import pandas as pd
from math import radians, sin, cos, sqrt, atan2

# ==================================================
# LOAD FILES
# ==================================================

stations = pd.read_excel(
    "east_anglia_master_v6.xlsx"
)

landmarks = pd.read_excel(
    "east_anglia_landmarks_v5.xlsx"
)

# ==================================================
# HAVERSINE DISTANCE
# ==================================================

def haversine(lat1, lon1, lat2, lon2):

    R = 6371.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        sin(dlat / 2) ** 2
        + cos(lat1)
        * cos(lat2)
        * sin(dlon / 2) ** 2
    )

    c = 2 * atan2(
        sqrt(a),
        sqrt(1 - a)
    )

    return R * c

# ==================================================
# FIND NEAREST LANDMARK
# ==================================================

nearest_landmark = []
nearest_landmark_km = []
nearest_category = []

for _, station in stations.iterrows():

    best_name = None
    best_category = None
    best_distance = 999999

    for _, landmark in landmarks.iterrows():

        distance = haversine(
            station["latitude"],
            station["longitude"],
            landmark["latitude"],
            landmark["longitude"]
        )

        if distance < best_distance:

            best_distance = distance
            best_name = landmark["landmark_name"]
            best_category = landmark["category"]

    nearest_landmark.append(best_name)
    nearest_landmark_km.append(round(best_distance, 2))
    nearest_category.append(best_category)

# ==================================================
# ADD COLUMNS
# ==================================================

stations["nearest_landmark"] = nearest_landmark
stations["nearest_landmark_km"] = nearest_landmark_km
stations["nearest_landmark_category"] = nearest_category

# ==================================================
# SAVE
# ==================================================

stations.to_excel(
    "east_anglia_master_v7.xlsx",
    index=False
)

stations.to_csv(
    "east_anglia_master_v7.csv",
    index=False
)

# ==================================================
# QA OUTPUT
# ==================================================

print("Saved: east_anglia_master_v7.xlsx")
print()

print(
    stations[
        [
            "station_name",
            "nearest_landmark",
            "nearest_landmark_category",
            "nearest_landmark_km"
        ]
    ]
    .head(20)
)

print()
print(
    "Unique landmarks used:",
    stations["nearest_landmark"].nunique()
)

Saved: east_anglia_master_v7.xlsx

                 station_name        nearest_landmark  \
0                    Benfleet         Hadleigh Castle   
1                     Barking                O2 Arena   
2                   Brentwood           Hylands House   
3                    Basildon         Hadleigh Castle   
4                  Chelmsford    Chelmsford Cathedral   
5   Chafford Hundred Lakeside                O2 Arena   
6              Chadwell Heath                O2 Arena   
7                   Chalkwell  Southend Cliff Gardens   
8                  Colchester       Colchester Castle   
9               Dagenham Dock                O2 Arena   
10                       Diss         Wymondham Abbey   
11               Emerson Park                O2 Arena   
12               East Tilbury         Hadleigh Castle   
13                Forest Gate                O2 Arena   
14    London Fenchurch Street         Tower of London   
15                 Gidea Park                O2 Arena

In [ ]:
import pandas as pd
from math import radians, sin, cos, sqrt, atan2

# ==================================================
# LOAD
# ==================================================

stations = pd.read_excel(
    "east_anglia_master_v7.xlsx"
)

landmarks = pd.read_excel(
    "east_anglia_landmarks_v5.xlsx"
)

# ==================================================
# HAVERSINE
# ==================================================

def haversine(lat1, lon1, lat2, lon2):

    R = 6371.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        sin(dlat/2)**2
        + cos(lat1)
        * cos(lat2)
        * sin(dlon/2)**2
    )

    c = 2 * atan2(
        sqrt(a),
        sqrt(1-a)
    )

    return R * c

# ==================================================
# BAND FUNCTION
# ==================================================

def distance_band(km):

    if km <= 2:
        return "adjacent"

    if km <= 10:
        return "near"

    if km <= 25:
        return "close"

    if km <= 50:
        return "within_reach"

    return "distant"

# ==================================================
# CATEGORIES
# ==================================================

categories = {
    "castle": "Castle",
    "pier": "Pier",
    "cathedral": "Cathedral",
    "museum": "Museum",
    "country_house": "Country House",
    "nature_reserve": "Nature Reserve"
}

# ==================================================
# PROCESS EACH CATEGORY
# ==================================================

for short_name, category_name in categories.items():

    subset = landmarks[
        landmarks["category"] == category_name
    ].copy()

    nearest_name = []
    nearest_km = []
    nearest_band = []

    for _, station in stations.iterrows():

        best_name = None
        best_dist = 999999

        for _, landmark in subset.iterrows():

            dist = haversine(
                station["latitude"],
                station["longitude"],
                landmark["latitude"],
                landmark["longitude"]
            )

            if dist < best_dist:

                best_dist = dist
                best_name = landmark["landmark_name"]

        nearest_name.append(best_name)
        nearest_km.append(round(best_dist, 2))
        nearest_band.append(
            distance_band(best_dist)
        )

    stations[
        f"nearest_{short_name}"
    ] = nearest_name

    stations[
        f"nearest_{short_name}_km"
    ] = nearest_km

    stations[
        f"nearest_{short_name}_band"
    ] = nearest_band

# ==================================================
# SAVE
# ==================================================

stations.to_excel(
    "east_anglia_master_v8.xlsx",
    index=False
)

stations.to_csv(
    "east_anglia_master_v8.csv",
    index=False
)

print("Saved: east_anglia_master_v8.xlsx")

print()
print(
    stations[
        [
            "station_name",
            "nearest_castle",
            "nearest_castle_band",
            "nearest_pier",
            "nearest_pier_band"
        ]
    ].head(20)
)


Saved: east_anglia_master_v8.xlsx

                 station_name      nearest_castle nearest_castle_band  \
0                    Benfleet     Hadleigh Castle                near   
1                     Barking     Hadleigh Castle        within_reach   
2                   Brentwood     Hadleigh Castle               close   
3                    Basildon     Hadleigh Castle               close   
4                  Chelmsford     Hadleigh Castle               close   
5   Chafford Hundred Lakeside     Hadleigh Castle               close   
6              Chadwell Heath     Hadleigh Castle        within_reach   
7                   Chalkwell     Hadleigh Castle                near   
8                  Colchester   Colchester Castle            adjacent   
9               Dagenham Dock     Hadleigh Castle        within_reach   
10                       Diss  Framlingham Castle               close   
11               Emerson Park     Hadleigh Castle        within_reach   
12              

In [ ]:
import pandas as pd

df = pd.read_excel(
    "east_anglia_master_v8.xlsx"
)

# ==========================================
# CLUE GENERATOR
# ==========================================

def make_clue(name, band):

    if pd.isna(name):
        return None

    if band == "adjacent":
        return f"Adjacent to {name}"

    if band == "near":
        return f"Near {name}"

    if band == "close":
        return f"Close to {name}"

    if band == "within_reach":
        return f"Within reach of {name}"

    return None

# ==========================================
# CREATE CLUES
# ==========================================

categories = [
    "castle",
    "pier",
    "cathedral",
    "museum",
    "country_house",
    "nature_reserve"
]

for cat in categories:

    df[f"{cat}_clue"] = df.apply(
        lambda row: make_clue(
            row[f"nearest_{cat}"],
            row[f"nearest_{cat}_band"]
        ),
        axis=1
    )

# ==========================================
# SAVE
# ==========================================

df.to_excel(
    "east_anglia_master_v9.xlsx",
    index=False
)

print("Saved: east_anglia_master_v9.xlsx")

print()

print(
    df[
        [
            "station_name",
            "castle_clue",
            "pier_clue"
        ]
    ]
    .head(20)
)

Saved: east_anglia_master_v9.xlsx

                 station_name                      castle_clue  \
0                    Benfleet             Near Hadleigh Castle   
1                     Barking  Within reach of Hadleigh Castle   
2                   Brentwood         Close to Hadleigh Castle   
3                    Basildon         Close to Hadleigh Castle   
4                  Chelmsford         Close to Hadleigh Castle   
5   Chafford Hundred Lakeside         Close to Hadleigh Castle   
6              Chadwell Heath  Within reach of Hadleigh Castle   
7                   Chalkwell             Near Hadleigh Castle   
8                  Colchester    Adjacent to Colchester Castle   
9               Dagenham Dock  Within reach of Hadleigh Castle   
10                       Diss      Close to Framlingham Castle   
11               Emerson Park  Within reach of Hadleigh Castle   
12               East Tilbury         Close to Hadleigh Castle   
13                Forest Gate  Within rea

In [ ]:
import pandas as pd

df = pd.read_excel("east_anglia_master_v8.xlsx")

for station_name in [
    "Benfleet",
    "Colchester",
    "Southend Central",
    "Norwich",
    "London Fenchurch Street"
]:

    print("\n" + "=" * 80)
    print(station_name)
    print("=" * 80)

    station = df[
        df["station_name"] == station_name
    ]

    print(station.T)


Benfleet
                                                   0
station_id                                   c2c_020
station_name                                Benfleet
crs                                              BEF
operator                                         c2c
route                                  Southend Line
origin_station               London Fenchurch Street
origin_minutes                                    40
distance_from_origin_km                         44.4
terminus                                       False
interchange                                    False
country                                      England
county                               Southend-on-Sea
zone                                        Southend
distance_band                               regional
coastal                                         True
latitude                                   51.543964
longitude                                   0.561273
northing_rank                       

In [ ]:
df = pd.read_excel("east_anglia_master_v8.xlsx")

print(
    df[
        df["major_interchange"]
    ][
        ["station_name"]
    ]
)

               station_name
1                   Barking
8                Colchester
22                  Ipswich
27  London Liverpool Street
32                  Norwich
36                  Romford
37                  Romford
41                Shenfield
43                Stratford
50                 West Ham


In [ ]:
import pandas as pd

df = pd.read_excel("east_anglia_master_v8.xlsx")

df.loc[
    df["station_name"] == "London Fenchurch Street",
    "major_interchange"
] = True

df.to_excel(
    "east_anglia_master_v8a.xlsx",
    index=False
)

print("Updated")

Updated


In [ ]:
import pandas as pd

df = pd.read_excel("east_anglia_master_v8a.xlsx")

clue_tags = []

for _, row in df.iterrows():

    tags = []

    # Railway

    if row["terminus"]:
        tags.append("terminus")

    if row["major_interchange"]:
        tags.append("major_interchange")

    if row["branch_junction"]:
        tags.append("branch_junction")

    # Geography

    if row["coastal"]:
        tags.append("coastal")

    if row["seaside_resort"]:
        tags.append("seaside_resort")

    if row["airport_station"]:
        tags.append("airport")

    # Distance

    tags.append(row["distance_band"])

    # Landmark categories

    landmark_categories = [
        "castle",
        "pier",
        "cathedral",
        "museum",
        "country_house",
        "nature_reserve"
    ]

    for cat in landmark_categories:

        band = row.get(f"nearest_{cat}_band")

        if band != "distant":
            tags.append(cat)

    clue_tags.append(
        "|".join(sorted(set(tags)))
    )

df["clue_tags"] = clue_tags

df.to_excel(
    "east_anglia_master_v11.xlsx",
    index=False
)

print("Saved: east_anglia_master_v11.xlsx")
print()

for station in [
    "Benfleet",
    "Colchester",
    "Southend Central",
    "Norwich",
    "London Fenchurch Street"
]:

    print("\n" + "="*60)
    print(station)

    result = df[
        df["station_name"] == station
    ][
        ["station_name", "clue_tags"]
    ]

    print(result.to_string(index=False))

Saved: east_anglia_master_v11.xlsx


Benfleet
station_name                                                                  clue_tags
    Benfleet castle|cathedral|coastal|country_house|museum|nature_reserve|pier|regional

Colchester
station_name                                                                            clue_tags
  Colchester castle|cathedral|country_house|major_interchange|museum|nature_reserve|pier|regional

Southend Central
    station_name                                                                                 clue_tags
Southend Central castle|cathedral|coastal|country_house|museum|nature_reserve|pier|regional|seaside_resort

Norwich
station_name                                                                                 clue_tags
     Norwich castle|cathedral|country_house|long|major_interchange|museum|nature_reserve|pier|terminus

London Fenchurch Street
           station_name                                                                      clue_

In [ ]:
# Landmark categories

landmark_categories = [
    "castle",
    "pier",
    "cathedral",
    "museum",
    "country_house",
    "nature_reserve"
]

for cat in landmark_categories:

    band = row.get(f"nearest_{cat}_band")

    # Only include genuinely nearby landmarks
    if band in [
        "adjacent",
        "near",
        "close"
    ]:
        tags.append(cat)

In [ ]:
# Landmark categories

landmark_categories = [
    "castle",
    "pier",
    "cathedral",
    "museum",
    "country_house",
    "nature_reserve"
]

for cat in landmark_categories:

    band = row.get(f"nearest_{cat}_band")

    # Only include genuinely nearby landmarks
    if band in [
        "adjacent",
        "near",
        "close"
    ]:
        tags.append(cat)

In [ ]:
import pandas as pd

df = pd.read_excel("east_anglia_master_v8a.xlsx")

clue_tags = []

for _, row in df.iterrows():

    tags = []

    # Railway

    if row["terminus"]:
        tags.append("terminus")

    if row["major_interchange"]:
        tags.append("major_interchange")

    if row["branch_junction"]:
        tags.append("branch_junction")

    # Geography

    if row["coastal"]:
        tags.append("coastal")

    if row["seaside_resort"]:
        tags.append("seaside_resort")

    if row["airport_station"]:
        tags.append("airport")

    # Distance band

    tags.append(row["distance_band"])

    # Landmark categories
    # Only if genuinely nearby

    landmark_categories = [
        "castle",
        "pier",
        "cathedral",
        "museum",
        "country_house",
        "nature_reserve"
    ]

    for cat in landmark_categories:

        band = row.get(f"nearest_{cat}_band")

        if band in [
            "adjacent",
            "near",
            "close"
        ]:
            tags.append(cat)

    clue_tags.append(
        "|".join(sorted(set(tags)))
    )

df["clue_tags"] = clue_tags

df.to_excel(
    "east_anglia_master_v12.xlsx",
    index=False
)

print("Saved: east_anglia_master_v12.xlsx")

print()

for station in [
    "Benfleet",
    "Colchester",
    "Southend Central",
    "Norwich",
    "London Fenchurch Street"
]:

    print("\n" + "=" * 60)
    print(station)

    result = df[
        df["station_name"] == station
    ][
        ["station_name", "clue_tags"]
    ]

    print(result.to_string(index=False))

Saved: east_anglia_master_v12.xlsx


Benfleet
station_name                                                           clue_tags
    Benfleet castle|cathedral|coastal|country_house|nature_reserve|pier|regional

Colchester
station_name                                                    clue_tags
  Colchester castle|major_interchange|museum|nature_reserve|pier|regional

Southend Central
    station_name                                                  clue_tags
Southend Central castle|coastal|nature_reserve|pier|regional|seaside_resort

Norwich
station_name                                                                            clue_tags
     Norwich castle|cathedral|country_house|long|major_interchange|museum|nature_reserve|terminus

London Fenchurch Street
           station_name                                  clue_tags
London Fenchurch Street cathedral|local|major_interchange|terminus


In [ ]:
import pandas as pd

df = pd.read_excel(
    "east_anglia_master_v12.xlsx"
)

# ==========================================
# SCORING
# ==========================================

band_score = {
    "adjacent": 5,
    "near": 4,
    "close": 3,
    "within_reach": 1,
    "distant": 0
}

# ==========================================
# BUILD CLUES
# ==========================================

strengths = []
primary = []
secondary = []

categories = [
    "castle",
    "pier",
    "cathedral",
    "museum",
    "country_house",
    "nature_reserve"
]

for _, row in df.iterrows():

    score = 0

    primary_clues = []
    secondary_clues = []

    # -------------------------
    # Landmark clues
    # -------------------------

    for cat in categories:

        band = row.get(
            f"nearest_{cat}_band"
        )

        name = row.get(
            f"nearest_{cat}"
        )

        score += band_score.get(
            band,
            0
        )

        if band == "adjacent":

            primary_clues.append(
                f"Adjacent to {name}"
            )

        elif band == "near":

            primary_clues.append(
                f"Near {name}"
            )

        elif band == "close":

            secondary_clues.append(
                f"Close to {name}"
            )

    # -------------------------
    # Coastal
    # -------------------------

    if row["coastal"]:

        score += 4

        primary_clues.append(
            "Coastal station"
        )

    # -------------------------
    # Seaside resort
    # -------------------------

    if row["seaside_resort"]:

        score += 4

        primary_clues.append(
            "Seaside resort station"
        )

    # -------------------------
    # Airport
    # -------------------------

    if row["airport_station"]:

        score += 5

        primary_clues.append(
            "Serves an airport"
        )

    # -------------------------
    # Major interchange
    # -------------------------

    if row["major_interchange"]:

        score += 4

        primary_clues.append(
            "Major interchange"
        )

    # -------------------------
    # Terminus
    # -------------------------

    if row["terminus"]:

        score += 5

        primary_clues.append(
            "Terminus station"
        )

    # -------------------------
    # Branch junction
    # -------------------------

    if row["branch_junction"]:

        score += 3

        secondary_clues.append(
            "Branch-line junction"
        )

    # -------------------------
    # Distance band
    # -------------------------

    if row["distance_band"] == "local":

        secondary_clues.append(
            "Close to London"
        )

    elif row["distance_band"] == "regional":

        secondary_clues.append(
            "Regional station"
        )

    elif row["distance_band"] == "long":

        secondary_clues.append(
            "Long-distance station"
        )

    strengths.append(score)

    primary.append(
        " | ".join(
            sorted(set(primary_clues))
        )
    )

    secondary.append(
        " | ".join(
            sorted(set(secondary_clues))
        )
    )

# ==========================================
# STORE
# ==========================================

df["clue_strength"] = strengths
df["primary_clues"] = primary
df["secondary_clues"] = secondary

# ==========================================
# SAVE
# ==========================================

df.to_excel(
    "east_anglia_master_v13.xlsx",
    index=False
)

df.to_csv(
    "east_anglia_master_v13.csv",
    index=False
)

# ==========================================
# QA
# ==========================================

print(
    "Saved: east_anglia_master_v13.xlsx"
)

print()

print(
    df[
        [
            "station_name",
            "clue_strength"
        ]
    ]
    .sort_values(
        "clue_strength",
        ascending=False
    )
    .head(15)
)

print()

for station in [
    "Benfleet",
    "Colchester",
    "Southend Central",
    "Norwich",
    "London Fenchurch Street"
]:

    print("\n" + "=" * 70)
    print(station)

    sample = df[
        df["station_name"] == station
    ][
        [
            "clue_strength",
            "primary_clues",
            "secondary_clues"
        ]
    ]

    print(sample.to_string(index=False))

Saved: east_anglia_master_v13.xlsx

          station_name  clue_strength
32             Norwich             31
76      Clacton-on-Sea             28
25        Leigh-on-Sea             26
79  Walton-on-the-Naze             25
84        Harwich Town             25
44        Shoeburyness             24
8           Colchester             23
42    Southend Central             23
36             Romford             23
63    Southend Airport             23
49           Westcliff             22
7            Chalkwell             22
78      Frinton-on-Sea             21
59        Southminster             21
69           Braintree             21


Benfleet
 clue_strength                          primary_clues                                                                                                             secondary_clues
            21 Coastal station | Near Hadleigh Castle Close to Chelmsford Cathedral | Close to Hylands House | Close to RHS Hyde Hall | Close to Southend Pier | Region

In [ ]:
import pandas as pd
import random

df = pd.read_excel(
    "east_anglia_master_v13.xlsx"
)

# ==========================================
# BUILD CLUE POOLS
# ==========================================

def split_clues(text):

    if pd.isna(text):
        return []

    return [
        c.strip()
        for c in str(text).split("|")
        if c.strip()
    ]

# ==========================================
# GENERATE SAMPLE PUZZLES
# ==========================================

for i in range(20):

    station = df.sample(1).iloc[0]

    primary = split_clues(
        station["primary_clues"]
    )

    secondary = split_clues(
        station["secondary_clues"]
    )

    easy_pool = primary

    medium_pool = (
        primary + secondary
    )

    hard_pool = secondary

    easy = (
        random.choice(easy_pool)
        if easy_pool else "No clue"
    )

    medium = (
        random.choice(medium_pool)
        if medium_pool else "No clue"
    )

    hard = (
        random.choice(hard_pool)
        if hard_pool else "No clue"
    )

    print("=" * 70)

    print(
        f"Station: {station['station_name']}"
    )

    print(
        f"Easy   : {easy}"
    )

    print(
        f"Medium : {medium}"
    )

    print(
        f"Hard   : {hard}"
    )

    print(
        f"Score  : {station['clue_strength']}"
    )

    print()

Station: Ilford
Easy   : No clue
Medium : Close to London
Hard   : Close to St Paul's Cathedral
Score  : 7

Station: Harwich International
Easy   : Coastal station
Medium : Coastal station
Hard   : Long-distance station
Score  : 16

Station: Maryland
Easy   : Near St Paul's Cathedral
Medium : Near St Paul's Cathedral
Hard   : Close to London
Score  : 8

Station: Stowmarket
Easy   : No clue
Medium : Close to Framlingham Castle
Hard   : Close to Christchurch Mansion
Score  : 8

Station: Southminster
Easy   : Coastal station
Medium : Close to Southend Pier
Hard   : Close to Colchester Zoo
Score  : 21

Station: Limehouse
Easy   : Near St Paul's Cathedral
Medium : Close to London
Hard   : Close to London
Score  : 7

Station: Hythe
Easy   : Adjacent to Colchester Castle
Medium : Close to Clacton Pier
Hard   : Close to Clacton Pier
Score  : 19

Station: Manor Park
Easy   : No clue
Medium : Close to London
Hard   : Close to London
Score  : 7

Station: Limehouse
Easy   : Near St Paul's Cathedra

In [ ]:
import pandas as pd

df = pd.read_excel(
    "east_anglia_master_v13.xlsx"
)

# ==========================================
# HELPERS
# ==========================================

def add(clues, text):

    if text:
        clues.append(text)

# ==========================================
# BUILD CLUE POOLS
# ==========================================

easy_pool = []
medium_pool = []
hard_pool = []

for _, row in df.iterrows():

    easy = []
    medium = []
    hard = []

    # =====================================
    # CASTLE
    # =====================================

    band = row["nearest_castle_band"]

    if band in ["adjacent", "near"]:

        add(
            easy,
            f"{band.title()} {row['nearest_castle']}"
        )

    if band in ["adjacent", "near", "close"]:

        add(
            medium,
            "Near a castle"
        )

        add(
            hard,
            "Historic landmark nearby"
        )

    # =====================================
    # PIER
    # =====================================

    band = row["nearest_pier_band"]

    if band in ["adjacent", "near"]:

        add(
            easy,
            f"{band.title()} {row['nearest_pier']}"
        )

    if band in ["adjacent", "near", "close"]:

        add(
            medium,
            "Near a pier"
        )

        add(
            hard,
            "Coastal attraction nearby"
        )

    # =====================================
    # CATHEDRAL
    # =====================================

    band = row["nearest_cathedral_band"]

    if band in ["adjacent", "near"]:

        add(
            easy,
            f"{band.title()} {row['nearest_cathedral']}"
        )

    if band in ["adjacent", "near", "close"]:

        add(
            medium,
            "Near a cathedral"
        )

        add(
            hard,
            "Religious landmark nearby"
        )

    # =====================================
    # MUSEUM
    # =====================================

    band = row["nearest_museum_band"]

    if band in ["adjacent", "near"]:

        add(
            easy,
            f"{band.title()} {row['nearest_museum']}"
        )

    if band in ["adjacent", "near", "close"]:

        add(
            medium,
            "Near a museum"
        )

        add(
            hard,
            "Cultural venue nearby"
        )

    # =====================================
    # COUNTRY HOUSE
    # =====================================

    band = row["nearest_country_house_band"]

    if band in ["adjacent", "near"]:

        add(
            easy,
            f"{band.title()} {row['nearest_country_house']}"
        )

    if band in ["adjacent", "near", "close"]:

        add(
            medium,
            "Near a country house"
        )

        add(
            hard,
            "Historic estate nearby"
        )

    # =====================================
    # NATURE RESERVE
    # =====================================

    band = row["nearest_nature_reserve_band"]

    if band in ["adjacent", "near"]:

        add(
            easy,
            f"{band.title()} {row['nearest_nature_reserve']}"
        )

    if band in ["adjacent", "near", "close"]:

        add(
            medium,
            "Near a nature reserve"
        )

        add(
            hard,
            "Protected landscape nearby"
        )

    # =====================================
    # GEOGRAPHY
    # =====================================

    if row["coastal"]:

        easy.append(
            "Coastal station"
        )

        medium.append(
            "Near the coast"
        )

        hard.append(
            "Maritime location"
        )

    if row["seaside_resort"]:

        easy.append(
            "Seaside resort station"
        )

        medium.append(
            "Popular coastal destination"
        )

        hard.append(
            "Traditional British seaside town"
        )

    # =====================================
    # RAILWAY
    # =====================================

    if row["major_interchange"]:

        easy.append(
            "Major interchange"
        )

        medium.append(
            "Important connection station"
        )

        hard.append(
            "Key transport hub"
        )

    if row["airport_station"]:

        medium.append(
            "Airport connection"
        )

        hard.append(
            "Gateway to air travel"
        )

    if row["terminus"]:

        medium.append(
            "End of the line"
        )

        hard.append(
            "Route endpoint"
        )

    if row["branch_junction"]:

        medium.append(
            "Branch-line junction"
        )

        hard.append(
            "Gateway to another route"
        )

    # =====================================
    # DISTANCE
    # =====================================

    if row["distance_band"] == "local":

        hard.append(
            "Inner suburban station"
        )

    elif row["distance_band"] == "regional":

        hard.append(
            "Regional route station"
        )

    elif row["distance_band"] == "long":

        hard.append(
            "Long-distance route station"
        )

    # =====================================
    # SAVE
    # =====================================

    easy_pool.append(
        " | ".join(sorted(set(easy)))
    )

    medium_pool.append(
        " | ".join(sorted(set(medium)))
    )

    hard_pool.append(
        " | ".join(sorted(set(hard)))
    )

# ==========================================
# STORE
# ==========================================

df["easy_clue_pool"] = easy_pool
df["medium_clue_pool"] = medium_pool
df["hard_clue_pool"] = hard_pool

# ==========================================
# SAVE
# ==========================================

df.to_excel(
    "east_anglia_master_v14.xlsx",
    index=False
)

print(
    "Saved: east_anglia_master_v14.xlsx"
)

print()

for station in [
    "Norwich",
    "Southend Central",
    "Colchester",
    "Benfleet",
    "London Fenchurch Street"
]:

    print("\n" + "=" * 70)
    print(station)

    sample = df[
        df["station_name"] == station
    ][
        [
            "easy_clue_pool",
            "medium_clue_pool",
            "hard_clue_pool"
        ]
    ]

    print(sample.to_string(index=False))

Saved: east_anglia_master_v14.xlsx


Norwich
                                                                                       easy_clue_pool                                                                                                                                 medium_clue_pool                                                                                                                                                                                        hard_clue_pool
Adjacent Norwich Castle | Adjacent Norwich Cathedral | Adjacent The Forum Norwich | Major interchange End of the line | Important connection station | Near a castle | Near a cathedral | Near a country house | Near a museum | Near a nature reserve Cultural venue nearby | Historic estate nearby | Historic landmark nearby | Key transport hub | Long-distance route station | Protected landscape nearby | Religious landmark nearby | Route endpoint

Southend Central
                                               

In [ ]:
import shutil

files_to_backup = [
    "east_anglia_master_v14.xlsx",
    "east_anglia_master_v13.xlsx",
    "east_anglia_master_v12.xlsx",
    "east_anglia_master_v11.xlsx",
    "east_anglia_master_v10.xlsx",
    "east_anglia_master_v9.xlsx",
    "east_anglia_master_v8.xlsx",
    "east_anglia_landmarks_v5.xlsx"
]

for file in files_to_backup:

    backup_name = file.replace(
        ".xlsx",
        "_RELEASE.xlsx"
    )

    shutil.copy(
        file,
        backup_name
    )

    print(
        f"Created: {backup_name}"
    )

print("\nRelease backups complete.")

Created: east_anglia_master_v14_RELEASE.xlsx
Created: east_anglia_master_v13_RELEASE.xlsx
Created: east_anglia_master_v12_RELEASE.xlsx
Created: east_anglia_master_v11_RELEASE.xlsx


FileNotFoundError: [Errno 2] No such file or directory: 'east_anglia_master_v10.xlsx'

In [ ]:
import os

for file in sorted(os.listdir()):
    print(file)

.config
c2c_master.xlsx
east_anglia_landmarks_v1.xlsx
east_anglia_landmarks_v2.xlsx
east_anglia_landmarks_v3.xlsx
east_anglia_landmarks_v4.xlsx
east_anglia_landmarks_v5.xlsx
east_anglia_master_v11.xlsx
east_anglia_master_v11_RELEASE.xlsx
east_anglia_master_v12.xlsx
east_anglia_master_v12_RELEASE.xlsx
east_anglia_master_v13.csv
east_anglia_master_v13.xlsx
east_anglia_master_v13_RELEASE.xlsx
east_anglia_master_v14.xlsx
east_anglia_master_v14_RELEASE.xlsx
east_anglia_master_v1_backup.xlsx
east_anglia_master_v2.xlsx
east_anglia_master_v3.xlsx
east_anglia_master_v4.csv
east_anglia_master_v4.xlsx
east_anglia_master_v5.csv
east_anglia_master_v5.xlsx
east_anglia_master_v5_RELEASE.csv
east_anglia_master_v5_RELEASE.xlsx
east_anglia_master_v6.xlsx
east_anglia_master_v7.csv
east_anglia_master_v7.xlsx
east_anglia_master_v8.csv
east_anglia_master_v8.xlsx
east_anglia_master_v8a.xlsx
east_anglia_master_v9.xlsx
greateranglia_extensions.xlsx
greateranglia_extensions_fixed.xlsx
greateranglia_extensions_v

In [ ]:
import shutil

files = [
    "east_anglia_master_v14.xlsx",
    "east_anglia_landmarks_v5.xlsx",
    "east_anglia_master_v13.csv"
]

for f in files:

    backup = f.replace(
        ".xlsx",
        "_FINAL_BACKUP.xlsx"
    ).replace(
        ".csv",
        "_FINAL_BACKUP.csv"
    )

    shutil.copy(f, backup)

    print("Created:", backup)

Created: east_anglia_master_v14_FINAL_BACKUP.xlsx
Created: east_anglia_landmarks_v5_FINAL_BACKUP.xlsx
Created: east_anglia_master_v13_FINAL_BACKUP.csv


In [ ]:
import pandas as pd

df = pd.read_excel("chiltern_midlands_master_v1.xlsx")

print(df.shape)
print(df.columns.tolist())
df.head()

(24, 23)
['station_id', 'station_name', 'crs', 'operator', 'route', 'county', 'latitude', 'longitude', 'major_interchange', 'terminus', 'branch_junction', 'nearest_landmark', 'nearest_landmark_km', 'nearest_castle', 'nearest_castle_km', 'nearest_cathedral', 'nearest_cathedral_km', 'nearest_museum', 'nearest_museum_km', 'nearest_country_house', 'nearest_country_house_km', 'nearest_nature_reserve', 'nearest_nature_reserve_km']


,station_id,station_name,crs,operator,route,county,latitude,longitude,major_interchange,terminus,...,nearest_castle,nearest_castle_km,nearest_cathedral,nearest_cathedral_km,nearest_museum,nearest_museum_km,nearest_country_house,nearest_country_house_km,nearest_nature_reserve,nearest_nature_reserve_km
0,CM001,London Marylebone,NaN,Chiltern Railways,Main Line,Greater London,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,CM002,Wembley Stadium,NaN,Chiltern Railways,Main Line,Greater London,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,CM003,South Ruislip,NaN,Chiltern Railways,Main Line,Greater London,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,CM004,West Ruislip,NaN,Chiltern Railways,Main Line,Greater London,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,CM005,Denham,NaN,Chiltern Railways,Main Line,Buckinghamshire,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
crs_lookup = {
    "London Marylebone": "MYB",
    "Wembley Stadium": "WCX",
    "South Ruislip": "SRU",
    "West Ruislip": "WRU",
    "Denham": "DNM",
    "Denham Golf Club": "DGC",
    "Gerrards Cross": "GER",
    "Seer Green & Jordans": "SRJ",
    "Beaconsfield": "BCF",
    "High Wycombe": "HWY",
    "Saunderton": "SDR",
    "Princes Risborough": "PRR",
    "Haddenham & Thame Parkway": "HDM",
    "Bicester Village": "BIT",
    "Kings Sutton": "KGS",
    "Banbury": "BAN",
    "Leamington Spa": "LMS",
    "Warwick": "WRW",
    "Warwick Parkway": "WRP",
    "Hatton": "HTN",
    "Dorridge": "DDG",
    "Solihull": "SOL",
    "Birmingham Moor Street": "BMO",
    "Birmingham Snow Hill": "BSW"
}

df["crs"] = df["station_name"].map(crs_lookup)

print(df[["station_name", "crs"]])

                 station_name  crs
0           London Marylebone  MYB
1             Wembley Stadium  WCX
2               South Ruislip  SRU
3                West Ruislip  WRU
4                      Denham  DNM
5            Denham Golf Club  DGC
6              Gerrards Cross  GER
7        Seer Green & Jordans  SRJ
8                Beaconsfield  BCF
9                High Wycombe  HWY
10                 Saunderton  SDR
11         Princes Risborough  PRR
12  Haddenham & Thame Parkway  HDM
13           Bicester Village  BIT
14               Kings Sutton  KGS
15                    Banbury  BAN
16             Leamington Spa  LMS
17                    Warwick  WRW
18            Warwick Parkway  WRP
19                     Hatton  HTN
20                   Dorridge  DDG
21                   Solihull  SOL
22     Birmingham Moor Street  BMO
23       Birmingham Snow Hill  BSW


In [ ]:
coords_lookup = {
    "London Marylebone": (51.5225, -0.1631),
    "Wembley Stadium": (51.5566, -0.2796),
    "South Ruislip": (51.5569, -0.3991),
    "West Ruislip": (51.5697, -0.4378),
    "Denham": (51.5785, -0.5178),
    "Denham Golf Club": (51.5809, -0.5170),
    "Gerrards Cross": (51.5890, -0.5550),
    "Seer Green & Jordans": (51.6110, -0.6070),
    "Beaconsfield": (51.6117, -0.6434),
    "High Wycombe": (51.6290, -0.7450),
    "Saunderton": (51.6750, -0.8250),
    "Princes Risborough": (51.7250, -0.8440),
    "Haddenham & Thame Parkway": (51.7700, -0.9420),
    "Bicester Village": (51.8930, -1.1490),
    "Kings Sutton": (52.0180, -1.2810),
    "Banbury": (52.0600, -1.3290),
    "Leamington Spa": (52.2850, -1.5350),
    "Warwick": (52.2860, -1.5820),
    "Warwick Parkway": (52.2860, -1.6130),
    "Hatton": (52.2950, -1.6720),
    "Dorridge": (52.3720, -1.7530),
    "Solihull": (52.4140, -1.7890),
    "Birmingham Moor Street": (52.4790, -1.8930),
    "Birmingham Snow Hill": (52.4830, -1.8990),
}


In [ ]:
coords_lookup = {
    "London Marylebone": (51.5225, -0.1631),
    "Wembley Stadium": (51.5566, -0.2796),
    "South Ruislip": (51.5569, -0.3991),
    "West Ruislip": (51.5697, -0.4378),
    "Denham": (51.5785, -0.5178),
    "Denham Golf Club": (51.5809, -0.5170),
    "Gerrards Cross": (51.5890, -0.5550),
    "Seer Green & Jordans": (51.6110, -0.6070),
    "Beaconsfield": (51.6117, -0.6434),
    "High Wycombe": (51.6290, -0.7450),
    "Saunderton": (51.6750, -0.8250),
    "Princes Risborough": (51.7250, -0.8440),
    "Haddenham & Thame Parkway": (51.7700, -0.9420),
    "Bicester Village": (51.8930, -1.1490),
    "Kings Sutton": (52.0180, -1.2810),
    "Banbury": (52.0600, -1.3290),
    "Leamington Spa": (52.2850, -1.5350),
    "Warwick": (52.2860, -1.5820),
    "Warwick Parkway": (52.2860, -1.6130),
    "Hatton": (52.2950, -1.6720),
    "Dorridge": (52.3720, -1.7530),
    "Solihull": (52.4140, -1.7890),
    "Birmingham Moor Street": (52.4790, -1.8930),
    "Birmingham Snow Hill": (52.4830, -1.8990),
}

In [ ]:
df.to_excel(
    "chiltern_midlands_master_v3.xlsx",
    index=False
)

In [ ]:
import pandas as pd

stations = pd.read_excel("chiltern_midlands_master_v3.xlsx")
landmarks = pd.read_excel("chiltern_midlands_landmarks_v2.xlsx")

FileNotFoundError: [Errno 2] No such file or directory: 'chiltern_midlands_landmarks_v2.xlsx'

In [ ]:
import pandas as pd

stations = pd.read_excel("chiltern_midlands_master_v3.xlsx")
landmarks = pd.read_excel("chiltern_midlands_landmarks_v2.xlsx")

In [ ]:
print(landmarks["category"].value_counts())

category
landmark          25
country_house     16
nature_reserve    13
museum            12
castle             9
cathedral          5
Name: count, dtype: int64


In [ ]:
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        sin(dlat / 2) ** 2
        + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    )

    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    return R * c

In [ ]:
def nearest_location(station_row, landmark_df):

    nearest_name = None
    nearest_distance = 999999

    for _, landmark in landmark_df.iterrows():

        distance = haversine(
            station_row["latitude"],
            station_row["longitude"],
            landmark["latitude"],
            landmark["longitude"]
        )

        if distance < nearest_distance:
            nearest_distance = distance
            nearest_name = landmark["landmark_name"]

    return nearest_name, round(nearest_distance, 2)

In [ ]:
stations["nearest_landmark"] = ""
stations["nearest_landmark_km"] = None

for idx, row in stations.iterrows():

    name, distance = nearest_location(
        row,
        landmarks
    )

    stations.loc[idx, "nearest_landmark"] = name
    stations.loc[idx, "nearest_landmark_km"] = distance

In [ ]:
stations[
    ["station_name",
     "nearest_landmark",
     "nearest_landmark_km"]
].head(10)

,station_name,nearest_landmark,nearest_landmark_km
0,London Marylebone,None,999999
1,Wembley Stadium,None,999999
2,South Ruislip,None,999999
3,West Ruislip,None,999999
4,Denham,None,999999
5,Denham Golf Club,None,999999
6,Gerrards Cross,None,999999
7,Seer Green & Jordans,None,999999
8,Beaconsfield,None,999999
9,High Wycombe,None,999999


In [ ]:
stations[
    ["station_name","latitude","longitude"]
].head(10)

,station_name,latitude,longitude
0,London Marylebone,NaN,NaN
1,Wembley Stadium,NaN,NaN
2,South Ruislip,NaN,NaN
3,West Ruislip,NaN,NaN
4,Denham,NaN,NaN
5,Denham Golf Club,NaN,NaN
6,Gerrards Cross,NaN,NaN
7,Seer Green & Jordans,NaN,NaN
8,Beaconsfield,NaN,NaN
9,High Wycombe,NaN,NaN


In [ ]:
print(df["station_name"].head().tolist())

['London Marylebone', 'Wembley Stadium', 'South Ruislip', 'West Ruislip', 'Denham']


In [ ]:
print(coords_lookup.get("London Marylebone"))

(51.5225, -0.1631)


In [ ]:
print(df["station_name"].iloc[0])
print(repr(df["station_name"].iloc[0]))
print(coords_lookup.get(df["station_name"].iloc[0]))

London Marylebone
'London Marylebone'
(51.5225, -0.1631)


In [ ]:
df["station_name"] = df["station_name"].astype(str).str.strip()

df["latitude"] = df["station_name"].apply(
    lambda x: coords_lookup[x][0]
)

df["longitude"] = df["station_name"].apply(
    lambda x: coords_lookup[x][1]
)

print(df[["station_name","latitude","longitude"]].head())
print()
print("Missing lat:", df["latitude"].isna().sum())
print("Missing lon:", df["longitude"].isna().sum())

        station_name  latitude  longitude
0  London Marylebone   51.5225    -0.1631
1    Wembley Stadium   51.5566    -0.2796
2      South Ruislip   51.5569    -0.3991
3       West Ruislip   51.5697    -0.4378
4             Denham   51.5785    -0.5178

Missing lat: 0
Missing lon: 0


In [ ]:
df.to_excel(
    "chiltern_midlands_master_v3.xlsx",
    index=False
)

In [ ]:
import pandas as pd

stations = pd.read_excel("chiltern_midlands_master_v3.xlsx")
landmarks = pd.read_excel("chiltern_midlands_landmarks_v2.xlsx")

In [ ]:
print(
    landmarks[
        ["landmark_name","latitude","longitude"]
    ].head()
)

       landmark_name  latitude  longitude
0     Warwick Castle   52.2797    -1.5849
1  Kenilworth Castle   52.3490    -1.5780
2      Dudley Castle   52.5158    -2.0829
3    Tamworth Castle   52.6340    -1.6950
4  Shrewsbury Castle   52.7106    -2.7496


In [ ]:
stations[
    [
        "station_name",
        "nearest_landmark",
        "nearest_landmark_km"
    ]
].head(15)

,station_name,nearest_landmark,nearest_landmark_km
0,London Marylebone,NaN,NaN
1,Wembley Stadium,NaN,NaN
2,South Ruislip,NaN,NaN
3,West Ruislip,NaN,NaN
4,Denham,NaN,NaN
5,Denham Golf Club,NaN,NaN
6,Gerrards Cross,NaN,NaN
7,Seer Green & Jordans,NaN,NaN
8,Beaconsfield,NaN,NaN
9,High Wycombe,NaN,NaN


In [ ]:
print(stations[["station_name","latitude","longitude"]].head())

        station_name  latitude  longitude
0  London Marylebone   51.5225    -0.1631
1    Wembley Stadium   51.5566    -0.2796
2      South Ruislip   51.5569    -0.3991
3       West Ruislip   51.5697    -0.4378
4             Denham   51.5785    -0.5178


In [ ]:
test_station = stations.iloc[0]

name, dist = nearest_location(
    test_station,
    landmarks
)

print("Nearest:", name)
print("Distance:", dist)

Nearest: Bekonscot Model Village
Distance: 34.64


In [ ]:
stations["nearest_landmark"] = ""
stations["nearest_landmark_km"] = 0.0

for idx, row in stations.iterrows():

    name, dist = nearest_location(row, landmarks)

    stations.at[idx, "nearest_landmark"] = name
    stations.at[idx, "nearest_landmark_km"] = dist

print("Done")

Done


In [ ]:
stations[
    [
        "station_name",
        "nearest_landmark",
        "nearest_landmark_km"
    ]
].head(10)


,station_name,nearest_landmark,nearest_landmark_km
0,London Marylebone,Bekonscot Model Village,34.64
1,Wembley Stadium,Bekonscot Model Village,25.89
2,South Ruislip,Bekonscot Model Village,17.92
3,West Ruislip,Bekonscot Model Village,14.93
4,Denham,Bekonscot Model Village,9.38
5,Denham Golf Club,Bekonscot Model Village,9.34
6,Gerrards Cross,Bekonscot Model Village,6.57
7,Seer Green & Jordans,Bekonscot Model Village,2.65
8,Beaconsfield,Bekonscot Model Village,0.43
9,High Wycombe,Hughenden Manor,2.56


In [ ]:
categories = [
    "castle",
    "cathedral",
    "museum",
    "country_house",
    "nature_reserve"
]

for category in categories:

    subset = landmarks[
        landmarks["category"] == category
    ]

    stations[f"nearest_{category}"] = ""
    stations[f"nearest_{category}_km"] = 0.0

    for idx, row in stations.iterrows():

        name, dist = nearest_location(
            row,
            subset
        )

        stations.at[idx, f"nearest_{category}"] = name
        stations.at[idx, f"nearest_{category}_km"] = dist

    print(f"{category} complete")

castle complete
cathedral complete
museum complete
country_house complete
nature_reserve complete


In [ ]:
stations[
    [
        "station_name",
        "nearest_castle",
        "nearest_cathedral",
        "nearest_museum",
        "nearest_country_house",
        "nearest_nature_reserve"
    ]
].head(10)

,station_name,nearest_castle,nearest_cathedral,nearest_museum,nearest_country_house,nearest_nature_reserve
0,London Marylebone,Oxford Castle,Coventry Cathedral,Roald Dahl Museum,National Trust Cliveden,Chiltern Hills
1,Wembley Stadium,Oxford Castle,Coventry Cathedral,Roald Dahl Museum,National Trust Cliveden,Chiltern Hills
2,South Ruislip,Oxford Castle,Coventry Cathedral,Roald Dahl Museum,National Trust Cliveden,Chiltern Hills
3,West Ruislip,Oxford Castle,Coventry Cathedral,Roald Dahl Museum,National Trust Cliveden,Chiltern Hills
4,Denham,Oxford Castle,Coventry Cathedral,Roald Dahl Museum,National Trust Cliveden,Chiltern Hills
5,Denham Golf Club,Oxford Castle,Coventry Cathedral,Roald Dahl Museum,National Trust Cliveden,Chiltern Hills
6,Gerrards Cross,Oxford Castle,Coventry Cathedral,Roald Dahl Museum,National Trust Cliveden,Chiltern Hills
7,Seer Green & Jordans,Oxford Castle,Coventry Cathedral,Roald Dahl Museum,National Trust Cliveden,Chiltern Hills
8,Beaconsfield,Oxford Castle,Coventry Cathedral,Roald Dahl Museum,National Trust Cliveden,Chiltern Hills
9,High Wycombe,Oxford Castle,Coventry Cathedral,Roald Dahl Museum,Hughenden Manor,Chiltern Hills


In [ ]:
stations.shape
stations.columns.tolist()

['station_id',
 'station_name',
 'crs',
 'operator',
 'route',
 'county',
 'latitude',
 'longitude',
 'major_interchange',
 'terminus',
 'branch_junction',
 'nearest_landmark',
 'nearest_landmark_km',
 'nearest_castle',
 'nearest_castle_km',
 'nearest_cathedral',
 'nearest_cathedral_km',
 'nearest_museum',
 'nearest_museum_km',
 'nearest_country_house',
 'nearest_country_house_km',
 'nearest_nature_reserve',
 'nearest_nature_reserve_km']

In [ ]:
stations.to_excel(
    "chiltern_midlands_master_v4.xlsx",
    index=False
)